In [13]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
#!/usr/bin/env python3
# ============================================================
# Con φ  ~  Report Data Preparation
# ============================================================
#
# VERSION BLOCK
# -------------
# Set these two variables once here. All paths and output
# filenames are derived from them automatically.
# If you are running this notebook standalone (not via the
# master runner), uncomment the lines below and set your own
# BASE_DIR.
#
VERSION = "v1"
BASE_DIR_DEFAULT = "/content/drive/MyDrive/conphi"
#
# ── Per-script override (uncomment if running standalone) ────
# from pathlib import Path
# VERSION  = "v1"
# BASE_DIR = Path("/content/drive/MyDrive/conphi")   # 👈 change to your path
# ─────────────────────────────────────────────────────────────
#
# If BASE_DIR is not already defined (i.e. running standalone
# without the master runner), fall back to the default above.
try:
    BASE_DIR
except NameError:
    from pathlib import Path
    BASE_DIR = Path(BASE_DIR_DEFAULT)

BASE_DIR = Path(BASE_DIR)

"""
Con φ v1 — Report Data Preparation
======================================

Reads prediction masters from both Con φ sub-models (USE and WASE),
harmonises column schemas with clean report-ready names, computes
metrics at multiple granularities, and writes parquet files for direct
report ingestion.

NAMING CONVENTIONS (report-facing)
--------------------------------------
  Model Type          USE  (Update Survey Estimate)
                      WASE (Without Any Survey Estimate)
  Prediction Type     Nowcast / Forecast
  Year                The focal/target year (the year we "stand at")
  Prediction Year     The actual year being predicted (= Year + Horizon)
  Data Type           Validated / Prediction Only
  Country             ISO3 code (display: Country Name)
  Region              World Bank region (6 groups)
  Sub-Region          World Bank sub-region (19 groups)

INPUTS
------
  conphi/data - model inputs/conphi_v1_features_{year}.parquet
  conphi/outputs/conphi_v1_use_{mode_suffix}/
  conphi/outputs/conphi_v1_wase_{mode_suffix}/

OUTPUT
------
  conphi/outputs/conphi_v1_report/
    fact_predictions.parquet
    fact_parameters.parquet
    dim_country.parquet
    dim_model_type.parquet
    dim_prediction_type.parquet
    dim_validation_mode.parquet
    diagnostics_*.parquet
    scatter_obs_vs_pred*.html
    manifest.json
"""

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json

warnings.filterwarnings("ignore")

# ============================================================
# PATHS  (all derived from BASE_DIR and VERSION)
# ============================================================
INPUT_DIR  = BASE_DIR / "data - model inputs"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUT_DIR    = OUTPUTS_DIR / f"conphi_{VERSION}_report"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model name prefixes — must match the model scripts
USE_MODEL_NAME  = f"conphi_{VERSION}_use"
WASE_MODEL_NAME = f"conphi_{VERSION}_wase"

LOG_CLIP = (-20.0, 20.0)

def safe_exp(x):
    return np.exp(np.clip(x, *LOG_CLIP))


# ============================================================
# WFP COUNTRY OPERATIONS LIST
# ============================================================
WFP_ISOS = {
    "AFG", "DZA", "ARM", "BGD", "BEN", "BTN", "BOL", "BFA", "BDI",
    "KHM", "CMR", "CAF", "TCD", "CHN", "COL", "COG", "CUB", "CIV",
    "COD", "DJI", "DOM", "PRK", "ECU", "EGY", "SLV", "SWZ", "ETH",
    "GMB", "GHA", "GTM", "GIN", "GNB", "HTI", "HND", "IND", "IDN",
    "IRN", "IRQ", "JOR", "KEN", "KGZ", "LAO", "LBN", "LSO", "LBR",
    "LBY", "MDG", "MWI", "MLI", "MRT", "MDA", "MOZ", "MMR", "NAM",
    "NPL", "NIC", "NER", "NGA", "PAK", "PSE", "PER", "PHL", "AGO",
    "RWA", "STP", "SEN", "SLE", "SOM", "SSD", "LKA", "SDN", "SYR",
    "TJK", "TZA", "TLS", "TGO", "TUN", "TUR", "UGA", "UKR", "VEN",
    "YEM", "ZMB", "ZWE",
}

def _load_income_groups():
    fp = INPUT_DIR / ".." / "data - raw" / "worldbank" / "wb_income_status_2024.csv"
    fp = fp.resolve()
    if not fp.exists():
        print("  WARNING: wb_income_status_2024.csv not found")
        return {}
    wb = pd.read_csv(fp)
    wb = wb.rename(columns={"Code": "iso", "Income group": "income_group"})
    wb["iso"]          = wb["iso"].astype(str).str.strip()
    wb["income_group"] = wb["income_group"].astype(str).str.strip()
    return wb.set_index("iso")["income_group"].to_dict()


# ============================================================
# COLUMN NAME MAPPING — single source of truth
# ============================================================
COL = {
    "country_code":                  "Country Code",
    "country_name":                  "Country",
    "year":                          "Year",
    "prediction_year":               "Prediction Year",
    "percentile":                    "Percentile",
    "horizon":                       "Horizon",
    "model_type":                    "Model Type",
    "prediction_type":               "Prediction Type",
    "data_type":                     "Data Type",
    "region":                        "Region",
    "sub_region":                    "Sub-Region",
    "anchor_year":                   "Anchor Year",
    "dt":                            "Years Since Survey",
    "observed_log_consumption":      "Observed Log Consumption",
    "observed_consumption":          "Observed Consumption",
    "predicted_log_consumption":     "Predicted Log Consumption",
    "predicted_consumption":         "Predicted Consumption",
    "lower_band_log":                "Lower Band Log (5th)",
    "upper_band_log":                "Upper Band Log (95th)",
    "lower_band":                    "Lower Band (5th)",
    "upper_band":                    "Upper Band (95th)",
    "lower_predictive_band_log":     "Lower Predictive Band Log (5th)",
    "upper_predictive_band_log":     "Upper Predictive Band Log (95th)",
    "lower_predictive_band":         "Lower Predictive Band (5th)",
    "upper_predictive_band":         "Upper Predictive Band (95th)",
    "residual":                      "Residual",
    "absolute_error":                "Absolute Error",
    "percentage_error":              "Percentage Error",
    "log_residual":                  "Log Residual",
    "log_absolute_error":            "Log Absolute Error",
    "azure_name":                    "Azure Name",
    "income_group":                  "Income Group",
    "wfp_country":                   "WFP Country",
    "country_scope":                 "Country Scope",
}

METRIC_COLS = {
    "n_obs":              "Observations",
    "n_countries":        "Countries",
    "mae_cons":           "MAE (Consumption)",
    "rmse_cons":          "RMSE (Consumption)",
    "bias_cons":          "Bias (Consumption)",
    "mape_pct":           "MAPE %",
    "r2_cons":            "R² (Consumption)",
    "mae_log":            "MAE (Log)",
    "rmse_log":           "RMSE (Log)",
    "bias_log":           "Bias (Log)",
    "r2_log":             "R² (Log)",
    "coverage_90_mu":     "Coverage 90% (Mean)",
    "coverage_90_y":      "Coverage 90% (Predictive)",
    "mahler_loss":        "Mahler Loss (×100)",
    "mean_pct_bias":      "Mean % Bias",
    "median_pct_bias":    "Median % Bias",
}

# ============================================================
# PARAMETER NAME MAPPING
# ============================================================
PARAM_MAP_USE = {
    "beta0_pos":  "Expansion Passthrough (Avg)",
    "beta0_neg":  "Contraction Passthrough (Avg)",
    "beta_p_pos": "Expansion Distributional Tilt",
    "beta_p_neg": "Contraction Distributional Tilt",
    "sigma":      "Observation Noise Scale",
    "nu":         "Degrees of Freedom (Tails)",
}

PARAM_MAP_WASE = {
    "intercept":              "Global Baseline Consumption",
    "gdp_elasticity":         "GDP Elasticity",
    "gdp_curvature":          "GDP Curvature (Non-linear)",
    "u5_mortality_effect":    "Under-5 Mortality Effect",
    "rural_share_effect":     "Rural Population Share Effect",
    "female_education_effect": "Female Education Effect",
    "gdp_growth_effect":      "GDP Growth Effect",
    "baseline_inequality":    "Baseline Inequality (Shape)",
    "noise_baseline":         "Observation Noise (Base)",
    "noise_tail_widening":    "Observation Noise (Tails)",
}


# ============================================================
# COUNTRY DIMENSION
# ============================================================
def build_country_dimension():
    """Build country dimension from most recent Con φ feature file."""
    geo_df = None
    for yr in range(2025, 2009, -1):
        vp = INPUT_DIR / f"conphi_{VERSION}_features_{yr}.parquet"
        if vp.exists():
            raw    = pd.read_parquet(vp)
            read_cols = ["iso"] + [c for c in ["region", "region23"] if c in raw.columns]
            geo_df = raw[read_cols].drop_duplicates("iso").copy()
            print(f"  Country dimension: {len(geo_df)} ISOs from vintage {yr}")
            break

    if geo_df is None:
        print("  WARNING: No feature file found for country dimension.")
        return pd.DataFrame()

    centroids = _get_centroids()
    geo_df    = geo_df.merge(centroids, on="iso", how="left")

    rename = {"iso": COL["country_code"], "country_name": COL["country_name"],
              "latitude": "Latitude", "longitude": "Longitude"}
    if "region"   in geo_df.columns: rename["region"]   = COL["region"]
    if "region23" in geo_df.columns: rename["region23"] = COL["sub_region"]

    geo_df = geo_df.rename(columns=rename)
    geo_df[COL["country_name"]] = geo_df[COL["country_name"]].fillna(geo_df[COL["country_code"]])
    return geo_df


def _get_centroids():
    data = {
        "AFG": ("Afghanistan", 33.0, 65.0), "ALB": ("Albania", 41.0, 20.0),
        "DZA": ("Algeria", 28.0, 3.0), "AGO": ("Angola", -12.5, 18.5),
        "ARG": ("Argentina", -34.0, -64.0), "ARM": ("Armenia", 40.0, 45.0),
        "AZE": ("Azerbaijan", 40.5, 47.5), "BGD": ("Bangladesh", 24.0, 90.0),
        "BLR": ("Belarus", 53.0, 28.0), "BEN": ("Benin", 9.5, 2.3),
        "BTN": ("Bhutan", 27.5, 90.5), "BOL": ("Bolivia", -17.0, -65.0),
        "BIH": ("Bosnia & Herzegovina", 44.0, 17.8),
        "BWA": ("Botswana", -22.0, 24.0), "BRA": ("Brazil", -10.0, -55.0),
        "BGR": ("Bulgaria", 42.7, 25.5), "BFA": ("Burkina Faso", 13.0, -1.5),
        "BDI": ("Burundi", -3.5, 29.9), "KHM": ("Cambodia", 13.0, 105.0),
        "CMR": ("Cameroon", 6.0, 12.0), "CAF": ("Central African Republic", 7.0, 21.0),
        "TCD": ("Chad", 15.0, 19.0), "CHL": ("Chile", -30.0, -71.0),
        "CHN": ("China", 35.0, 105.0), "COL": ("Colombia", 4.0, -72.0),
        "COM": ("Comoros", -12.2, 44.3), "COG": ("Congo, Rep.", -1.0, 15.0),
        "COD": ("Congo, Dem. Rep.", -3.0, 23.0), "CRI": ("Costa Rica", 10.0, -84.0),
        "CIV": ("Côte d'Ivoire", 8.0, -5.5), "HRV": ("Croatia", 45.2, 15.5),
        "CZE": ("Czech Republic", 49.8, 15.5), "DJI": ("Djibouti", 11.5, 43.0),
        "DOM": ("Dominican Republic", 19.0, -70.7),
        "ECU": ("Ecuador", -2.0, -77.5), "EGY": ("Egypt", 27.0, 30.0),
        "SLV": ("El Salvador", 13.8, -88.9), "GNQ": ("Equatorial Guinea", 2.0, 10.0),
        "ERI": ("Eritrea", 15.0, 39.0), "EST": ("Estonia", 59.0, 26.0),
        "SWZ": ("Eswatini", -26.5, 31.5), "ETH": ("Ethiopia", 8.0, 38.0),
        "FJI": ("Fiji", -18.0, 175.0), "GAB": ("Gabon", -1.0, 11.8),
        "GMB": ("Gambia", 13.5, -15.5), "GEO": ("Georgia", 42.0, 43.5),
        "GHA": ("Ghana", 8.0, -2.0), "GTM": ("Guatemala", 15.5, -90.3),
        "GIN": ("Guinea", 11.0, -10.0), "GNB": ("Guinea-Bissau", 12.0, -15.0),
        "GUY": ("Guyana", 5.0, -59.0), "HTI": ("Haiti", 19.0, -72.4),
        "HND": ("Honduras", 15.0, -86.5), "HUN": ("Hungary", 47.0, 20.0),
        "IND": ("India", 20.0, 77.0), "IDN": ("Indonesia", -5.0, 120.0),
        "IRN": ("Iran", 32.0, 53.0), "IRQ": ("Iraq", 33.0, 44.0),
        "JAM": ("Jamaica", 18.1, -77.3), "JOR": ("Jordan", 31.0, 36.0),
        "KAZ": ("Kazakhstan", 48.0, 68.0), "KEN": ("Kenya", 1.0, 38.0),
        "KGZ": ("Kyrgyz Republic", 41.0, 75.0), "LAO": ("Lao PDR", 18.0, 105.0),
        "LVA": ("Latvia", 57.0, 25.0), "LBN": ("Lebanon", 33.8, 35.8),
        "LSO": ("Lesotho", -29.5, 28.5), "LBR": ("Liberia", 6.5, -9.5),
        "LTU": ("Lithuania", 56.0, 24.0), "MDG": ("Madagascar", -20.0, 47.0),
        "MWI": ("Malawi", -13.5, 34.0), "MYS": ("Malaysia", 2.5, 112.5),
        "MDV": ("Maldives", 3.2, 73.2), "MLI": ("Mali", 17.0, -4.0),
        "MRT": ("Mauritania", 20.0, -12.0), "MUS": ("Mauritius", -20.3, 57.6),
        "MEX": ("Mexico", 23.0, -102.0), "MDA": ("Moldova", 47.0, 29.0),
        "MNG": ("Mongolia", 46.0, 105.0), "MNE": ("Montenegro", 42.5, 19.3),
        "MAR": ("Morocco", 32.0, -5.0), "MOZ": ("Mozambique", -18.0, 35.0),
        "MMR": ("Myanmar", 22.0, 98.0), "NAM": ("Namibia", -22.0, 17.0),
        "NPL": ("Nepal", 28.0, 84.0), "NIC": ("Nicaragua", 13.0, -85.0),
        "NER": ("Niger", 16.0, 8.0), "NGA": ("Nigeria", 10.0, 8.0),
        "MKD": ("North Macedonia", 41.5, 22.0), "PAK": ("Pakistan", 30.0, 70.0),
        "PAN": ("Panama", 9.0, -80.0), "PNG": ("Papua New Guinea", -6.0, 147.0),
        "PRY": ("Paraguay", -23.0, -58.0), "PER": ("Peru", -10.0, -76.0),
        "PHL": ("Philippines", 13.0, 122.0), "POL": ("Poland", 52.0, 20.0),
        "ROU": ("Romania", 46.0, 25.0), "RUS": ("Russia", 60.0, 100.0),
        "RWA": ("Rwanda", -2.0, 29.9), "STP": ("São Tomé & Príncipe", 1.0, 7.0),
        "SEN": ("Senegal", 14.0, -14.0), "SRB": ("Serbia", 44.0, 21.0),
        "SLE": ("Sierra Leone", 8.5, -11.5), "SVK": ("Slovakia", 48.7, 19.7),
        "SVN": ("Slovenia", 46.1, 14.8), "SLB": ("Solomon Islands", -8.0, 159.0),
        "SOM": ("Somalia", 5.0, 46.0), "ZAF": ("South Africa", -29.0, 24.0),
        "SSD": ("South Sudan", 7.0, 30.0), "LKA": ("Sri Lanka", 7.0, 81.0),
        "SDN": ("Sudan", 15.0, 30.0), "SUR": ("Suriname", 4.0, -56.0),
        "SYR": ("Syria", 35.0, 38.0), "TJK": ("Tajikistan", 39.0, 71.0),
        "TZA": ("Tanzania", -6.0, 35.0), "THA": ("Thailand", 15.0, 100.0),
        "TLS": ("Timor-Leste", -8.8, 126.0), "TGO": ("Togo", 8.0, 1.2),
        "TTO": ("Trinidad & Tobago", 10.5, -61.3), "TUN": ("Tunisia", 34.0, 9.0),
        "TUR": ("Turkey", 39.0, 35.0), "TKM": ("Turkmenistan", 40.0, 60.0),
        "UGA": ("Uganda", 1.0, 32.0), "UKR": ("Ukraine", 49.0, 32.0),
        "URY": ("Uruguay", -33.0, -56.0), "UZB": ("Uzbekistan", 41.0, 64.0),
        "VNM": ("Vietnam", 16.0, 108.0), "PSE": ("West Bank & Gaza", 32.0, 35.3),
        "YEM": ("Yemen", 15.0, 48.0), "ZMB": ("Zambia", -15.0, 28.0),
        "ZWE": ("Zimbabwe", -20.0, 30.0), "CPV": ("Cabo Verde", 16.0, -24.0),
        "XKX": ("Kosovo", 42.6, 20.9), "VUT": ("Vanuatu", -16.0, 167.0),
        "TON": ("Tonga", -20.0, -175.0), "WSM": ("Samoa", -13.8, -172.0),
        "KIR": ("Kiribati", 1.4, 173.0), "FSM": ("Micronesia", 6.9, 158.2),
        "MHL": ("Marshall Islands", 7.1, 171.2), "PLW": ("Palau", 7.5, 134.6),
        "NRU": ("Nauru", -0.5, 166.9), "TUV": ("Tuvalu", -8.5, 179.2),
        "VEN": ("Venezuela", 8.0, -66.0), "LBY": ("Libya", 25.0, 17.0),
        "BLZ": ("Belize",       17.2, -88.7), "BRB": ("Barbados",     13.2, -59.6),
        "GRD": ("Grenada",      12.1, -61.7), "LCA": ("Saint Lucia",  13.9, -60.9),
        "SYC": ("Seychelles",   -4.7,  55.5),
    }
    rows = [{"iso": k, "country_name": v[0], "latitude": v[1], "longitude": v[2]}
            for k, v in data.items()]
    return pd.DataFrame(rows)


def _azure_name_lookup():
    return {
        "AFG": "Afghanistan", "ALB": "Albania", "DZA": "Algeria",
        "AGO": "Angola", "ARG": "Argentina", "ARM": "Armenia",
        "AZE": "Azerbaijan", "BGD": "Bangladesh", "BLR": "Belarus",
        "BEN": "Benin", "BTN": "Bhutan", "BOL": "Bolivia",
        "BIH": "Bosnia and Herzegovina", "BWA": "Botswana", "BRA": "Brazil",
        "BGR": "Bulgaria", "BFA": "Burkina Faso", "BDI": "Burundi",
        "KHM": "Cambodia", "CMR": "Cameroon",
        "CAF": "Central African Republic", "TCD": "Chad",
        "CHL": "Chile", "CHN": "China", "COL": "Colombia",
        "COM": "Comoros", "COG": "Republic of the Congo",
        "COD": "Democratic Republic of the Congo",
        "CRI": "Costa Rica", "CIV": "Ivory Coast", "HRV": "Croatia",
        "CZE": "Czech Republic", "DJI": "Djibouti",
        "DOM": "Dominican Republic", "ECU": "Ecuador", "EGY": "Egypt",
        "SLV": "El Salvador", "GNQ": "Equatorial Guinea",
        "ERI": "Eritrea", "EST": "Estonia", "SWZ": "Eswatini",
        "ETH": "Ethiopia", "FJI": "Fiji", "GAB": "Gabon",
        "GMB": "Gambia", "GEO": "Georgia", "GHA": "Ghana",
        "GTM": "Guatemala", "GIN": "Guinea", "GNB": "Guinea-Bissau",
        "GUY": "Guyana", "HTI": "Haiti", "HND": "Honduras",
        "HUN": "Hungary", "IND": "India", "IDN": "Indonesia",
        "IRN": "Iran", "IRQ": "Iraq", "JAM": "Jamaica",
        "JOR": "Jordan", "KAZ": "Kazakhstan", "KEN": "Kenya",
        "KGZ": "Kyrgyzstan", "LAO": "Laos", "LVA": "Latvia",
        "LBN": "Lebanon", "LSO": "Lesotho", "LBR": "Liberia",
        "LTU": "Lithuania", "MDG": "Madagascar", "MWI": "Malawi",
        "MYS": "Malaysia", "MDV": "Maldives", "MLI": "Mali",
        "MRT": "Mauritania", "MUS": "Mauritius", "MEX": "Mexico",
        "MDA": "Moldova", "MNG": "Mongolia", "MNE": "Montenegro",
        "MAR": "Morocco", "MOZ": "Mozambique", "MMR": "Myanmar",
        "NAM": "Namibia", "NPL": "Nepal", "NIC": "Nicaragua",
        "NER": "Niger", "NGA": "Nigeria", "MKD": "North Macedonia",
        "PAK": "Pakistan", "PAN": "Panama",
        "PNG": "Papua New Guinea", "PRY": "Paraguay", "PER": "Peru",
        "PHL": "Philippines", "POL": "Poland", "ROU": "Romania",
        "RUS": "Russia", "RWA": "Rwanda",
        "STP": "Sao Tome and Principe", "SEN": "Senegal",
        "SRB": "Serbia", "SLE": "Sierra Leone", "SVK": "Slovakia",
        "SVN": "Slovenia", "SLB": "Solomon Islands", "SOM": "Somalia",
        "ZAF": "South Africa", "SSD": "South Sudan",
        "LKA": "Sri Lanka", "SDN": "Sudan", "SUR": "Suriname",
        "SYR": "Syria", "TJK": "Tajikistan", "TZA": "Tanzania",
        "THA": "Thailand", "TLS": "Timor-Leste", "TGO": "Togo",
        "TTO": "Trinidad and Tobago", "TUN": "Tunisia",
        "TUR": "Turkey", "TKM": "Turkmenistan", "UGA": "Uganda",
        "UKR": "Ukraine", "URY": "Uruguay", "UZB": "Uzbekistan",
        "VNM": "Vietnam", "PSE": "Palestinian Territories",
        "YEM": "Yemen", "ZMB": "Zambia", "ZWE": "Zimbabwe",
        "CPV": "Cabo Verde", "VUT": "Vanuatu", "VEN": "Venezuela",
        "LBY": "Libya", "XKX": "Kosovo", "TON": "Tonga",
        "WSM": "Samoa", "KIR": "Kiribati", "FSM": "Micronesia",
        "MHL": "Marshall Islands", "PLW": "Palau", "NRU": "Nauru",
        "TUV": "Tuvalu",
        "BLZ": "Belize",
        "BRB": "Barbados",
        "GRD": "Grenada",
        "LCA": "Saint Lucia",
        "SYC": "Seychelles",
    }


# ============================================================
# SHARED PREDICTION BUILDER
# ============================================================
def _build_residuals(out):
    """Add pre-computed residual columns to a prediction DataFrame."""
    obs  = out[COL["observed_consumption"]].values
    pred = out[COL["predicted_consumption"]].values
    out[COL["residual"]]       = pred - obs
    out[COL["absolute_error"]] = np.abs(out[COL["residual"]])
    with np.errstate(divide="ignore", invalid="ignore"):
        pct_err = np.abs((pred - obs) / obs) * 100
        out[COL["percentage_error"]] = np.where(np.isfinite(pct_err), pct_err, np.nan)
    obs_log  = out[COL["observed_log_consumption"]].values
    pred_log = out[COL["predicted_log_consumption"]].values
    out[COL["log_residual"]]       = pred_log - obs_log
    out[COL["log_absolute_error"]] = np.abs(out[COL["log_residual"]])
    return out


def _apply_wase_ci_shrink(out, shrink=0.55):
    """Post-hoc CI recalibration for WASE (corrects SVI variance underestimation)."""
    mu_log = out[COL["predicted_log_consumption"]].values
    for lo_col, hi_col in [
        (COL["lower_band_log"],            COL["upper_band_log"]),
        (COL["lower_predictive_band_log"], COL["upper_predictive_band_log"]),
    ]:
        out[lo_col] = mu_log + shrink * (out[lo_col].values - mu_log)
        out[hi_col] = mu_log + shrink * (out[hi_col].values - mu_log)
    for log_col, cons_col in [
        (COL["lower_band_log"],            COL["lower_band"]),
        (COL["upper_band_log"],            COL["upper_band"]),
        (COL["lower_predictive_band_log"], COL["lower_predictive_band"]),
        (COL["upper_predictive_band_log"], COL["upper_predictive_band"]),
    ]:
        out[cons_col] = safe_exp(out[log_col].values)
    return out


# ============================================================
# DATA LOADING — WASE
# ============================================================
def load_wase_predictions(mode_suffix):
    """Load WASE rolling LOCO master predictions."""
    master_dir = OUTPUTS_DIR / f"{WASE_MODEL_NAME}_{mode_suffix}" / "master"
    path       = master_dir / f"{WASE_MODEL_NAME}_rolling_predictions_master_{mode_suffix}.parquet"
    if not path.exists():
        return None

    df = pd.read_parquet(path)
    print(f"  Loaded WASE rolling ({mode_suffix}): {len(df):,} rows")

    horizons = df["horizon"].astype(int).values
    out = pd.DataFrame()
    out[COL["country_code"]]    = df["iso"].values
    out[COL["year"]]            = df["focal_year"].astype(int).values
    out[COL["prediction_year"]] = df["year"].astype(int).values
    out[COL["percentile"]]      = df["percentile"].astype(int).values
    out[COL["horizon"]]         = horizons
    out[COL["model_type"]]      = "WASE"
    out[COL["prediction_type"]] = np.where(horizons == 0, "Nowcast", "Forecast")
    out[COL["region"]]          = df["region"].values   if "region"   in df.columns else np.nan
    out[COL["sub_region"]]      = df["region23"].values if "region23" in df.columns else np.nan
    out[COL["anchor_year"]]     = np.nan
    out[COL["dt"]]              = np.nan

    out[COL["observed_log_consumption"]]  = df["log_cons"].values
    out[COL["predicted_log_consumption"]] = df["mu_mean"].values
    out[COL["lower_band_log"]]            = df["mu_q05"].values
    out[COL["upper_band_log"]]            = df["mu_q95"].values
    out[COL["lower_predictive_band_log"]] = df["y_q05"].values
    out[COL["upper_predictive_band_log"]] = df["y_q95"].values

    out[COL["observed_consumption"]]  = np.where(
        out[COL["observed_log_consumption"]].notna(),
        safe_exp(out[COL["observed_log_consumption"]]), np.nan,
    )
    out[COL["predicted_consumption"]] = safe_exp(out[COL["predicted_log_consumption"]])
    out[COL["lower_band"]]            = safe_exp(out[COL["lower_band_log"]])
    out[COL["upper_band"]]            = safe_exp(out[COL["upper_band_log"]])
    out[COL["lower_predictive_band"]] = safe_exp(out[COL["lower_predictive_band_log"]])
    out[COL["upper_predictive_band"]] = safe_exp(out[COL["upper_predictive_band_log"]])

    out[COL["data_type"]] = np.where(
        out[COL["observed_consumption"]].notna(), "Validated", "Prediction Only"
    )
    out = _build_residuals(out)
    out = _apply_wase_ci_shrink(out)
    print(f"    WASE CI shrink applied (factor=0.55)")
    return out


def load_wase_production(mode_suffix):
    """Load WASE Phase 3 production predictions (all countries, all years)."""
    prod_dir = OUTPUTS_DIR / f"{WASE_MODEL_NAME}_{mode_suffix}" / "production"
    path     = prod_dir / f"{WASE_MODEL_NAME}_production_predictions_{mode_suffix}.parquet"
    if not path.exists():
        return None

    df = pd.read_parquet(path)
    print(f"  Loaded WASE production ({mode_suffix}): {len(df):,} rows")

    out = pd.DataFrame()
    out[COL["country_code"]]    = df["iso"].values
    out[COL["year"]]            = df["year"].astype(int).values
    out[COL["prediction_year"]] = df["year"].astype(int).values
    out[COL["percentile"]]      = df["percentile"].astype(int).values
    out[COL["horizon"]]         = 0
    out[COL["model_type"]]      = "WASE"
    out[COL["prediction_type"]] = "Nowcast"
    out[COL["region"]]          = df["region"].values   if "region"   in df.columns else np.nan
    out[COL["sub_region"]]      = df["region23"].values if "region23" in df.columns else np.nan
    out[COL["anchor_year"]]     = np.nan
    out[COL["dt"]]              = np.nan

    out[COL["observed_log_consumption"]]  = df["log_cons"].values if "log_cons" in df.columns else np.nan
    out[COL["predicted_log_consumption"]] = df["mu_mean"].values
    out[COL["lower_band_log"]]            = df["mu_q05"].values
    out[COL["upper_band_log"]]            = df["mu_q95"].values
    out[COL["lower_predictive_band_log"]] = df["y_q05"].values
    out[COL["upper_predictive_band_log"]] = df["y_q95"].values

    out[COL["observed_consumption"]]  = np.where(
        out[COL["observed_log_consumption"]].notna(),
        safe_exp(out[COL["observed_log_consumption"]]), np.nan,
    )
    out[COL["predicted_consumption"]] = safe_exp(out[COL["predicted_log_consumption"]])
    out[COL["lower_band"]]            = safe_exp(out[COL["lower_band_log"]])
    out[COL["upper_band"]]            = safe_exp(out[COL["upper_band_log"]])
    out[COL["lower_predictive_band"]] = safe_exp(out[COL["lower_predictive_band_log"]])
    out[COL["upper_predictive_band"]] = safe_exp(out[COL["upper_predictive_band_log"]])

    out[COL["data_type"]] = np.where(
        df["has_survey"] == True, "Validated", "Prediction Only"
    )
    out = _build_residuals(out)
    out = _apply_wase_ci_shrink(out)
    return out


# ============================================================
# DATA LOADING — USE
# ============================================================
def load_use_predictions(mode_suffix):
    """Load USE rolling LOCO master predictions."""
    save_dir = OUTPUTS_DIR / f"{USE_MODEL_NAME}_{mode_suffix}"
    path     = save_dir / f"{USE_MODEL_NAME}_predictions_master_{mode_suffix}.parquet"
    if not path.exists():
        return None

    df = pd.read_parquet(path)
    print(f"  Loaded USE ({mode_suffix}): {len(df):,} rows")

    horizons = df["horizon"].astype(int).values
    out = pd.DataFrame()
    out[COL["country_code"]]    = df["iso"].values
    out[COL["year"]]            = df["target_year"].astype(int).values
    out[COL["prediction_year"]] = df["year"].astype(int).values
    out[COL["percentile"]]      = df["percentile"].astype(int).values
    out[COL["horizon"]]         = horizons
    out[COL["model_type"]]      = "USE"
    out[COL["prediction_type"]] = np.where(horizons == 0, "Nowcast", "Forecast")
    out[COL["region"]]          = df["region"].values   if "region"   in df.columns else np.nan
    out[COL["sub_region"]]      = df["region23"].values if "region23" in df.columns else np.nan
    out[COL["anchor_year"]]     = df["anchor_year"].astype(int).values if "anchor_year" in df.columns else np.nan
    out[COL["dt"]]              = df["dt"].astype(int).values           if "dt"          in df.columns else np.nan

    out[COL["predicted_log_consumption"]] = df["log_cons_hat"].values
    out[COL["observed_log_consumption"]]  = (
        df["log_cons_actual"].values if "log_cons_actual" in df.columns else np.nan
    )
    out[COL["lower_band_log"]]            = df["mu_q05"].values
    out[COL["upper_band_log"]]            = df["mu_q95"].values
    out[COL["lower_predictive_band_log"]] = df["y_q05"].values
    out[COL["upper_predictive_band_log"]] = df["y_q95"].values

    out[COL["observed_consumption"]]  = np.where(
        out[COL["observed_log_consumption"]].notna(),
        safe_exp(out[COL["observed_log_consumption"]]), np.nan,
    )
    out[COL["predicted_consumption"]] = safe_exp(out[COL["predicted_log_consumption"]])
    out[COL["lower_band"]]            = safe_exp(out[COL["lower_band_log"]])
    out[COL["upper_band"]]            = safe_exp(out[COL["upper_band_log"]])
    out[COL["lower_predictive_band"]] = safe_exp(out[COL["lower_predictive_band_log"]])
    out[COL["upper_predictive_band"]] = safe_exp(out[COL["upper_predictive_band_log"]])

    out[COL["data_type"]] = np.where(
        out[COL["observed_consumption"]].notna(), "Validated", "Prediction Only"
    )
    out = _build_residuals(out)
    return out


# ============================================================
# PARAMETER LOADING
# ============================================================
def load_wase_parameters(mode_suffix):
    """Load WASE posterior parameter summaries from master parquet."""
    master_dir = OUTPUTS_DIR / f"{WASE_MODEL_NAME}_{mode_suffix}" / "master"
    path       = master_dir / f"{WASE_MODEL_NAME}_rolling_params_master_{mode_suffix}.parquet"
    if not path.exists():
        return pd.DataFrame()

    df  = pd.read_parquet(path)
    out = pd.DataFrame()
    out["Model Type"]          = "WASE"
    out["Prediction Type"]     = "Nowcast" if "nowcast" in mode_suffix else "Forecast"
    out["Target Year"]         = df["focal_year"].astype(int)
    out["Raw Parameter"]       = df["parameter"]
    out["Parameter Name"]      = df["parameter"].map(PARAM_MAP_WASE).fillna(df["parameter"])
    out["Mean"]                = df["posterior_mean"]
    out["Lower Bound"]         = df["posterior_q05"]
    out["Upper Bound"]         = df["posterior_q95"]
    out["Standard Deviation"]  = df["posterior_sd"]
    return out


def load_use_parameters(mode_suffix):
    """Load USE posterior summaries from per-year CSV files."""
    save_dir   = OUTPUTS_DIR / f"{USE_MODEL_NAME}_{mode_suffix}"
    if not save_dir.exists():
        return pd.DataFrame()

    param_rows = []
    for csv_path in save_dir.rglob(f"target_year=*/{USE_MODEL_NAME}_posterior_summary_*.csv"):
        year_str = csv_path.parent.name.split("=")[-1]
        try:
            year = int(year_str)
            df   = pd.read_csv(csv_path, index_col=0)
            for param, row in df.iterrows():
                if param in PARAM_MAP_USE:
                    param_rows.append({
                        "Model Type":       "USE",
                        "Prediction Type":  "Nowcast" if "nowcast" in mode_suffix else "Forecast",
                        "Target Year":      year,
                        "Parameter Name":   PARAM_MAP_USE[param],
                        "Raw Parameter":    param,
                        "Mean":             row.get("mean",    np.nan),
                        "Lower Bound":      row.get("hdi_3%",  np.nan),
                        "Upper Bound":      row.get("hdi_97%", np.nan),
                        "Standard Deviation": row.get("sd",   np.nan),
                    })
        except Exception as e:
            print(f"    Error reading {csv_path.name}: {e}")
    return pd.DataFrame(param_rows)


# ============================================================
# METRIC COMPUTATION
# ============================================================
def compute_metrics(df):
    C  = COL
    ev = df.dropna(subset=[
        C["observed_consumption"], C["predicted_consumption"],
        C["observed_log_consumption"], C["predicted_log_consumption"],
    ]).copy()
    if len(ev) == 0:
        return _empty_metrics()

    c_true  = ev[C["observed_consumption"]].values
    c_hat   = ev[C["predicted_consumption"]].values
    c_resid = c_hat - c_true
    ss_res_c = np.sum(c_resid ** 2)
    ss_tot_c = np.sum((c_true - c_true.mean()) ** 2)

    with np.errstate(divide="ignore", invalid="ignore"):
        ape      = np.abs(c_resid / c_true)
        ape      = np.where(np.isfinite(ape), ape, np.nan)
        pct_bias = (c_resid / c_true) * 100
        pct_bias = np.where(np.isfinite(pct_bias), pct_bias, np.nan)

    l_true  = ev[C["observed_log_consumption"]].values
    l_hat   = ev[C["predicted_log_consumption"]].values
    l_resid = l_hat - l_true
    ss_res_l = np.sum(l_resid ** 2)
    ss_tot_l = np.sum((l_true - l_true.mean()) ** 2)

    cov_mu = np.mean(
        (l_true >= ev[C["lower_band_log"]].values) &
        (l_true <= ev[C["upper_band_log"]].values)
    ) * 100
    cov_y  = np.mean(
        (l_true >= ev[C["lower_predictive_band_log"]].values) &
        (l_true <= ev[C["upper_predictive_band_log"]].values)
    ) * 100

    mahler = float(
        ev.assign(_abs_err=np.abs(l_resid))
        .groupby(C["country_code"])["_abs_err"].mean().mean() * 100
    )

    return {
        METRIC_COLS["n_obs"]:           int(len(ev)),
        METRIC_COLS["n_countries"]:     int(ev[C["country_code"]].nunique()),
        METRIC_COLS["mae_cons"]:        float(np.mean(np.abs(c_resid))),
        METRIC_COLS["rmse_cons"]:       float(np.sqrt(np.mean(c_resid ** 2))),
        METRIC_COLS["bias_cons"]:       float(c_resid.mean()),
        METRIC_COLS["mape_pct"]:        float(np.nanmean(ape) * 100),
        METRIC_COLS["r2_cons"]:         float(1.0 - ss_res_c / ss_tot_c) if ss_tot_c > 1e-8 else np.nan,
        METRIC_COLS["mae_log"]:         float(np.mean(np.abs(l_resid))),
        METRIC_COLS["rmse_log"]:        float(np.sqrt(np.mean(l_resid ** 2))),
        METRIC_COLS["bias_log"]:        float(l_resid.mean()),
        METRIC_COLS["r2_log"]:          float(1.0 - ss_res_l / ss_tot_l) if ss_tot_l > 1e-8 else np.nan,
        METRIC_COLS["coverage_90_mu"]:  float(cov_mu),
        METRIC_COLS["coverage_90_y"]:   float(cov_y),
        METRIC_COLS["mahler_loss"]:     float(mahler),
        METRIC_COLS["mean_pct_bias"]:   float(np.nanmean(pct_bias)),
        METRIC_COLS["median_pct_bias"]: float(np.nanmedian(pct_bias)),
    }


def _empty_metrics():
    return {v: (0 if k in ("n_obs", "n_countries") else np.nan) for k, v in METRIC_COLS.items()}


def compute_grouped_metrics(df, group_cols):
    rows = []
    for keys, grp in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        m   = compute_metrics(grp)
        row = dict(zip(group_cols, keys))
        row.update(m)
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


# ============================================================
# DIAGNOSTIC TABLE BUILDER
# ============================================================
def build_diagnostics(df, model_label, country_dim):
    C  = COL
    PT = C["prediction_type"]

    ev = df[df[C["data_type"]] == "Validated"].copy()
    if len(ev) == 0:
        print(f"    No validated rows for {model_label}")
        return {}

    tables = {}
    tables["overall"]     = compute_grouped_metrics(ev, [C["model_type"], PT])
    tables["by_year"]     = compute_grouped_metrics(ev, [C["model_type"], PT, C["year"]])

    tbl = compute_grouped_metrics(ev, [C["model_type"], PT, C["country_code"]])
    if len(tbl) > 0 and len(country_dim) > 0:
        name_map = country_dim.set_index(C["country_code"])[C["country_name"]].to_dict()
        tbl[C["country_name"]] = tbl[C["country_code"]].map(name_map).fillna(tbl[C["country_code"]])
    tables["by_country"]  = tbl

    tables["by_percentile"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["percentile"]])

    if C["region"] in ev.columns and ev[C["region"]].notna().any():
        tables["by_region"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["region"]])

    if C["sub_region"] in ev.columns and ev[C["sub_region"]].notna().any():
        tables["by_subregion"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["sub_region"]])

    tbl = compute_grouped_metrics(ev, [C["model_type"], PT, C["country_code"], C["percentile"]])
    if len(tbl) > 0 and len(country_dim) > 0:
        name_map = country_dim.set_index(C["country_code"])[C["country_name"]].to_dict()
        tbl[C["country_name"]] = tbl[C["country_code"]].map(name_map).fillna(tbl[C["country_code"]])
    tables["by_country_percentile"] = tbl

    if C["income_group"] in ev.columns and ev[C["income_group"]].notna().any():
        tables["by_income"]            = compute_grouped_metrics(ev, [C["model_type"], PT, C["income_group"]])
        tables["by_income_percentile"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["income_group"], C["percentile"]])
        tables["by_income_year"]       = compute_grouped_metrics(ev, [C["model_type"], PT, C["income_group"], C["year"]])
    if C["country_scope"] in ev.columns:
        tables["by_country_scope"]            = compute_grouped_metrics(ev, [C["model_type"], PT, C["country_scope"]])
        tables["by_country_scope_percentile"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["country_scope"], C["percentile"]])
    if C["wfp_country"] in ev.columns:
        tables["by_wfp"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["wfp_country"]])
    if model_label == "use" and C["dt"] in ev.columns and ev[C["dt"]].notna().any():
        tables["by_dt"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["dt"]])
        if C["income_group"] in ev.columns:
            tables["by_dt_income"] = compute_grouped_metrics(ev, [C["model_type"], PT, C["dt"], C["income_group"]])

    return tables


# ============================================================
# INTERACTIVE SCATTER — OBSERVED vs PREDICTED
# ============================================================
def build_scatter_html(df, out_dir):
    """Build Plotly observed-vs-predicted scatter with 45° line."""
    try:
        import plotly.graph_objects as go
    except ImportError:
        print("  WARNING: plotly not installed — skipping scatter HTML")
        return

    C  = COL
    ev = df[df[C["data_type"]] == "Validated"].copy()
    if len(ev) == 0:
        print("  No validated rows for scatter plot")
        return

    MODEL_COLOURS = {"USE": "#1f77b4", "WASE": "#ff7f0e"}

    def _make_scatter(plot_df, title, filename):
        fig = go.Figure()
        lo  = plot_df[[C["observed_consumption"], C["predicted_consumption"]]].min().min()
        hi  = plot_df[[C["observed_consumption"], C["predicted_consumption"]]].max().max()
        pad = (hi - lo) * 0.02
        fig.add_trace(go.Scatter(
            x=[lo - pad, hi + pad], y=[lo - pad, hi + pad],
            mode="lines", line=dict(color="rgba(100,100,100,0.5)", width=1.5, dash="dash"),
            name="Perfect fit", hoverinfo="skip",
        ))
        for model_type in sorted(plot_df[C["model_type"]].unique()):
            sub    = plot_df[plot_df[C["model_type"]] == model_type]
            colour = MODEL_COLOURS.get(model_type, "#333")
            hover_text = (
                "<b>" + sub[C["country_name"]].astype(str) + "</b> (" + sub[C["country_code"]].astype(str) + ")<br>"
                + "Year: "       + sub[C["year"]].astype(int).astype(str) + "<br>"
                + "Percentile: " + sub[C["percentile"]].astype(int).astype(str) + "<br>"
                + "Observed: "   + sub[C["observed_consumption"]].map("{:.3f}".format) + "<br>"
                + "Predicted: "  + sub[C["predicted_consumption"]].map("{:.3f}".format) + "<br>"
                + "Error: "      + sub[C["percentage_error"]].map("{:.1f}%".format) + "<br>"
                + "Region: "     + sub[C["region"]].fillna("").astype(str)
            ).tolist()
            fig.add_trace(go.Scattergl(
                x=sub[C["observed_consumption"]], y=sub[C["predicted_consumption"]],
                mode="markers", marker=dict(size=3, color=colour, opacity=0.5),
                name=model_type, text=hover_text, hoverinfo="text",
            ))
        fig.update_layout(
            title=dict(text=title, font=dict(size=16)),
            xaxis_title="Observed Consumption (2017 PPP $/day)",
            yaxis_title="Predicted Consumption (2017 PPP $/day)",
            template="plotly_white", width=900, height=700,
            legend=dict(x=0.02, y=0.98),
            margin=dict(l=60, r=30, t=60, b=60),
        )
        fig.update_xaxes(constrain="domain")
        fig.update_yaxes(scaleanchor="x", scaleratio=1)
        fig.write_html(out_dir / filename, include_plotlyjs="cdn")
        print(f"    {filename}: {len(plot_df):,} points")

    _make_scatter(ev, "Observed vs Predicted — All Models", "scatter_obs_vs_pred.html")
    for mt in ev[C["model_type"]].unique():
        _make_scatter(ev[ev[C["model_type"]] == mt],
                      f"Observed vs Predicted — {mt}",
                      f"scatter_obs_vs_pred_{mt.lower()}.html")
    for pt in ev[C["prediction_type"]].unique():
        safe = pt.lower().replace(" ", "_")
        _make_scatter(ev[ev[C["prediction_type"]] == pt],
                      f"Observed vs Predicted — {pt}",
                      f"scatter_obs_vs_pred_{safe}.html")


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 80)
    print(f"Con φ {VERSION}  ~  REPORT DATA PREPARATION")
    print(f"Output: {OUT_DIR}")
    print("=" * 80)

    # ── Dimension tables ──────────────────────────────────────
    print("\nBuilding dimension tables...")
    country_dim = build_country_dimension()
    if len(country_dim) > 0:
        country_dim.to_parquet(OUT_DIR / "dim_country.parquet", index=False)
        print(f"  dim_country: {len(country_dim)} countries")

    pd.DataFrame([
        {COL["model_type"]: "USE",  "Description": "Update Survey Estimate — projects from most recent survey using GDP growth passthrough"},
        {COL["model_type"]: "WASE", "Description": "Without Any Survey Estimate — predicts from structural country indicators only"},
    ]).to_parquet(OUT_DIR / "dim_model_type.parquet", index=False)

    pd.DataFrame([
        {COL["prediction_type"]: "Nowcast",  "Description": "Predicts the focal year (horizon 0)"},
        {COL["prediction_type"]: "Forecast", "Description": "Predicts beyond the focal year using IMF WEO projections"},
    ]).to_parquet(OUT_DIR / "dim_prediction_type.parquet", index=False)

    pd.DataFrame([
        {"Validation Mode": "Validated", "Description": "Show only predictions with observed survey data"},
        {"Validation Mode": "All",       "Description": "Show all predictions (validated + prediction only)"},
    ]).to_parquet(OUT_DIR / "dim_validation_mode.parquet", index=False)
    print("  dim_model_type, dim_prediction_type, dim_validation_mode: done")

    # ── Load predictions ──────────────────────────────────────
    print("\nLoading prediction masters...")
    all_frames = []
    modes = ["nowcast", "forecast_1yr", "forecast_2yr", "forecast_3yr", "forecast_4yr", "forecast_5yr"]

    for mode in modes:
        for loader in [load_wase_predictions, load_wase_production, load_use_predictions]:
            df = loader(mode)
            if df is not None and len(df) > 0:
                all_frames.append(df)

    if not all_frames:
        print("\nNo prediction masters found. Run USE and WASE models first.")
        return

    unified = pd.concat(all_frames, ignore_index=True)

    dedup_cols = [COL["country_code"], COL["prediction_year"], COL["percentile"],
                  COL["model_type"], COL["prediction_type"]]
    n_before = len(unified)
    unified  = unified.drop_duplicates(subset=dedup_cols, keep="first").reset_index(drop=True)
    n_dropped = n_before - len(unified)
    if n_dropped > 0:
        print(f"  Deduplication: dropped {n_dropped:,} duplicate rows")

    # ── Enrich with dimension lookups ─────────────────────────
    if len(country_dim) > 0:
        name_map = country_dim.set_index(COL["country_code"])[COL["country_name"]].to_dict()
        unified[COL["country_name"]] = unified[COL["country_code"]].map(name_map).fillna(unified[COL["country_code"]])

    azure_map = _azure_name_lookup()
    unified[COL["azure_name"]] = unified[COL["country_code"]].map(azure_map).fillna(unified[COL["country_code"]])

    income_map = _load_income_groups()
    unified[COL["income_group"]] = unified[COL["country_code"]].map(income_map).fillna("Unknown")
    print(f"  Income groups : {unified[COL['income_group']].nunique()} ({sorted(unified[COL['income_group']].unique().tolist())})")

    unified[COL["wfp_country"]]   = np.where(unified[COL["country_code"]].isin(WFP_ISOS), "Yes",           "No")
    unified[COL["country_scope"]] = np.where(unified[COL["country_code"]].isin(WFP_ISOS), "WFP Countries", "All Countries")
    n_wfp = unified.loc[unified[COL["wfp_country"]] == "Yes", COL["country_code"]].nunique()
    print(f"  WFP countries : {n_wfp}")

    n_val  = (unified[COL["data_type"]] == "Validated").sum()
    n_pred = (unified[COL["data_type"]] == "Prediction Only").sum()
    print(f"\nFact table: {len(unified):,} total rows")
    print(f"  Model Types      : {sorted(unified[COL['model_type']].unique().tolist())}")
    print(f"  Prediction Types : {sorted(unified[COL['prediction_type']].unique().tolist())}")
    print(f"  Countries        : {unified[COL['country_code']].nunique()}")
    print(f"  Validated        : {n_val:,}")
    print(f"  Prediction Only  : {n_pred:,}")

    unified.to_parquet(OUT_DIR / "fact_predictions.parquet", index=False)
    print(f"\n  fact_predictions.parquet: {len(unified):,} rows")

    # ── Parameters ────────────────────────────────────────────
    print("\nExtracting model parameters...")
    all_params = []
    for mode in modes:
        for loader in [load_use_parameters, load_wase_parameters]:
            p = loader(mode)
            if not p.empty:
                all_params.append(p)

    if all_params:
        fact_parameters = pd.concat(all_params, ignore_index=True).drop_duplicates(
            subset=["Model Type", "Prediction Type", "Target Year", "Raw Parameter"]
        )
        fact_parameters.to_parquet(OUT_DIR / "fact_parameters.parquet", index=False)
        print(f"  fact_parameters.parquet: {len(fact_parameters):,} rows")
    else:
        print("  No parameter files found.")

    # ── Diagnostic tables ─────────────────────────────────────
    print("\nBuilding diagnostic tables...")
    all_diagnostics = {}
    for model_key in ["USE", "WASE"]:
        label    = model_key.lower()
        model_df = unified[unified[COL["model_type"]] == model_key].copy()
        if len(model_df) == 0:
            continue
        print(f"\n  {model_key} diagnostics:")
        tables = build_diagnostics(model_df, label, country_dim)
        for tbl_name, tbl_df in tables.items():
            if len(tbl_df) == 0:
                continue
            fname = f"diagnostics_{label}_{tbl_name}.parquet"
            tbl_df.to_parquet(OUT_DIR / fname, index=False)
            print(f"    {fname}: {len(tbl_df)} rows")
            all_diagnostics.setdefault(tbl_name, []).append(tbl_df)

    print("\n  Combined diagnostics:")
    for tbl_name, frames in all_diagnostics.items():
        combined = pd.concat(frames, ignore_index=True)
        fname    = f"diagnostics_combined_{tbl_name}.parquet"
        combined.to_parquet(OUT_DIR / fname, index=False)
        print(f"    {fname}: {len(combined)} rows")

    # ── Scatter HTML ──────────────────────────────────────────
    print("\nBuilding interactive scatter plots...")
    build_scatter_html(unified, OUT_DIR)

    # ── Summary print ─────────────────────────────────────────
    overall_path = OUT_DIR / "diagnostics_combined_overall.parquet"
    if overall_path.exists():
        overall = pd.read_parquet(overall_path)
        C = COL; MC = METRIC_COLS
        print(f"\n{'=' * 100}")
        print(f"  {'Model':<8s} {'Pred Type':<12s} {'Obs':>8s} {'R²':>8s} "
              f"{'MAE':>8s} {'RMSE':>8s} {'MAPE%':>8s} "
              f"{'Bias':>8s} {'Cov90%':>8s} {'Mahler':>8s}")
        print(f"  {'-' * 92}")
        for _, r in overall.iterrows():
            print(
                f"  {r[C['model_type']]:<8s} {r[C['prediction_type']]:<12s} "
                f"{int(r[MC['n_obs']]):>8,d} "
                f"{r[MC['r2_cons']]:>8.4f} "
                f"{r[MC['mae_cons']]:>8.4f} "
                f"{r[MC['rmse_cons']]:>8.4f} "
                f"{r[MC['mape_pct']]:>8.2f} "
                f"{r[MC['bias_cons']]:>8.4f} "
                f"{r[MC['coverage_90_y']]:>8.1f} "
                f"{r[MC['mahler_loss']]:>8.2f}"
            )
        print(f"{'=' * 100}")

    # ── Manifest ──────────────────────────────────────────────
    manifest = {
        "model_version":          VERSION,
        "output_dir":             str(OUT_DIR),
        "fact_table":             "fact_predictions.parquet",
        "parameter_table":        "fact_parameters.parquet",
        "total_rows":             int(len(unified)),
        "validated_rows":         int(n_val),
        "prediction_only_rows":   int(n_pred),
        "dimension_tables":       [
            "dim_country.parquet", "dim_model_type.parquet",
            "dim_prediction_type.parquet", "dim_validation_mode.parquet",
        ],
        "diagnostic_tables": sorted([f.name for f in OUT_DIR.glob("diagnostics_*.parquet")]),
        "column_names":      {k: v for k, v in COL.items()},
        "metric_names":      {k: v for k, v in METRIC_COLS.items()},
    }
    with open(OUT_DIR / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    n_files = len(list(OUT_DIR.glob("*.parquet")))
    print(f"\nDone — {n_files} parquet files written to {OUT_DIR}")


if __name__ == "__main__":
    main()

Con φ v1  ~  REPORT DATA PREPARATION
Output: /content/drive/MyDrive/conphi/outputs/conphi_v1_report

Building dimension tables...
  Country dimension: 123 ISOs from vintage 2025
  dim_country: 123 countries
  dim_model_type, dim_prediction_type, dim_validation_mode: done

Loading prediction masters...
  Loaded WASE rolling (forecast_1yr): 41,583 rows
    WASE CI shrink applied (factor=0.55)
  Loaded WASE production (forecast_1yr): 145,137 rows
  Loaded USE (forecast_1yr): 253,539 rows
  Deduplication: dropped 23,565 duplicate rows
  Income groups : 5 (['High income', 'Low income', 'Lower middle income', 'Upper middle income', 'nan'])
  WFP countries : 69

Fact table: 416,694 total rows
  Model Types      : ['USE', 'WASE']
  Prediction Types : ['Forecast', 'Nowcast']
  Countries        : 122
  Validated        : 81,975
  Prediction Only  : 334,719

  fact_predictions.parquet: 416,694 rows

Extracting model parameters...
  fact_parameters.parquet: 296 rows

Building diagnostic tables...


In [16]:
#!/usr/bin/env python3
# ============================================================
# Con φ  ~  USE Diagnostic Report
# ============================================================
#
# VERSION BLOCK
# -------------
# Set these two variables once here. All paths and output
# filenames are derived from them automatically.
# If you are running this notebook standalone (not via the
# master runner), uncomment the lines below and set your own
# BASE_DIR.
#
VERSION = "v1"
BASE_DIR_DEFAULT = "/content/drive/MyDrive/conphi"
#
# ── Per-script override (uncomment if running standalone) ────
# from pathlib import Path
# VERSION  = "v1"
# BASE_DIR = Path("/content/drive/MyDrive/conphi")   # 👈 change to your path
# ─────────────────────────────────────────────────────────────
#
# If BASE_DIR is not already defined (i.e. running standalone
# without the master runner), fall back to the default above.
try:
    BASE_DIR
except NameError:
    from pathlib import Path
    BASE_DIR = Path(BASE_DIR_DEFAULT)

BASE_DIR = Path(BASE_DIR)

"""
Con φ v1 — USE Model Diagnostic Report
========================================

Produces a self-contained interactive HTML report (Plotly) and
parquet summary tables for the Con φ USE sub-model.

All metrics are computed on validated rows only (rows that have
matched observed survey data).  The report reads from the unified
fact table produced by the report prep script.

INPUTS
------
  conphi/outputs/conphi_v1_report/fact_predictions.parquet
  conphi/outputs/conphi_v1_use_{mode}/target_year=*/
      conphi_v1_use_posterior_summary_{year}.csv

OUTPUTS
-------
  conphi/outputs/conphi_v1_report/diagnostics_use/
    use_diagnostics.html
    use_diag_params_over_time.parquet
    use_diag_residuals.parquet
    use_diag_coverage.parquet
    use_diag_country_mae.parquet
    use_diag_dt_metrics.parquet
    use_diag_region_metrics.parquet
    use_diag_overall_metrics.parquet
"""

import numpy as np
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# PATHS  (all derived from BASE_DIR and VERSION)
# ============================================================
OUTPUTS_DIR    = BASE_DIR / "outputs"
USE_MODEL_NAME = f"conphi_{VERSION}_use"
REPORT_DIR    = OUTPUTS_DIR / f"conphi_{VERSION}_report"
OUT_DIR        = REPORT_DIR / "diagnostics_use"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FACT_PATH = REPORT_DIR / "fact_predictions.parquet"

MODES = [
    "nowcast",
    "forecast_1yr", "forecast_2yr",
    "forecast_3yr", "forecast_4yr", "forecast_5yr",
]

PARAM_MAP_USE = {
    "beta0_pos":  "β⁺ Expansion Passthrough (Avg)",
    "beta0_neg":  "β⁻ Contraction Passthrough (Avg)",
    "beta_p_pos": "β_p⁺ Expansion Distributional Tilt",
    "beta_p_neg": "β_p⁻ Contraction Distributional Tilt",
    "sigma":      "σ Observation Noise Scale",
    "nu":         "ν Degrees of Freedom (Tails)",
}

INCOME_ORDER = [
    "Low income", "Lower middle income",
    "Upper middle income", "High income", "Unknown",
]

WB_PALETTE = {
    "East Asia & Pacific":        "#1f77b4",
    "Europe & Central Asia":      "#aec7e8",
    "Latin America & Caribbean":  "#ff7f0e",
    "Middle East & North Africa": "#ffbb78",
    "South Asia":                 "#2ca02c",
    "Sub-Saharan Africa":         "#d62728",
    "Unknown":                    "#999999",
}

INCOME_PALETTE = {
    "Low income":          "#d62728",
    "Lower middle income": "#ff7f0e",
    "Upper middle income": "#2ca02c",
    "High income":         "#1f77b4",
    "Unknown":             "#999999",
}


# ============================================================
# HELPERS
# ============================================================
LOG_CLIP = (-20.0, 20.0)


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def safe_exp(x):
    return np.exp(np.clip(x, *LOG_CLIP))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return float(1.0 - ss_res / ss_tot) if ss_tot > 1e-8 else np.nan


def compute_metrics_df(df):
    ev = df.dropna(subset=["obs_log", "pred_log"]).copy()
    if len(ev) == 0:
        return {}
    obs_l   = ev["obs_log"].values
    pred_l  = ev["pred_log"].values
    obs_c   = safe_exp(obs_l)
    pred_c  = safe_exp(pred_l)
    resid_l = pred_l - obs_l
    resid_c = pred_c - obs_c
    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.where(obs_c > 0, np.abs(resid_c / obs_c), np.nan)
    iso_col = "iso" if "iso" in ev.columns else None
    return {
        "n_obs":       int(len(ev)),
        "n_countries": int(ev[iso_col].nunique()) if iso_col else np.nan,
        "mae_log":     float(np.mean(np.abs(resid_l))),
        "rmse_log":    float(np.sqrt(np.mean(resid_l ** 2))),
        "bias_log":    float(resid_l.mean()),
        "r2_log":      r2_score(obs_l, pred_l),
        "mae_cons":    float(np.mean(np.abs(resid_c))),
        "rmse_cons":   float(np.sqrt(np.mean(resid_c ** 2))),
        "bias_cons":   float(resid_c.mean()),
        "r2_cons":     r2_score(obs_c, pred_c),
        "mape_pct":    float(np.nanmean(ape) * 100),
        "mahler":      float(
            ev.assign(_ae=np.abs(resid_l))
              .groupby(iso_col)["_ae"].mean().mean() * 100
        ) if iso_col else np.nan,
    }


# ============================================================
# DATA LOADING
# ============================================================
def load_use_fact():
    """Pull USE rows from the unified report fact table."""
    if not FACT_PATH.exists():
        raise FileNotFoundError(
            f"Fact table not found at {FACT_PATH}.\n"
            "Run conphi_v1_report_prep.py first."
        )
    df  = pd.read_parquet(FACT_PATH)
    use = df[df["Model Type"] == "USE"].copy()
    print(f"  USE rows from fact table: {len(use):,}")

    rename = {
        "Country Code":                     "iso",
        "Country":                          "country_name",
        "Year":                             "target_year",
        "Prediction Year":                  "pred_year",
        "Percentile":                       "percentile",
        "Horizon":                          "horizon",
        "Prediction Type":                  "pred_type",
        "Data Type":                        "data_type",
        "Region":                           "region",
        "Sub-Region":                       "sub_region",
        "Income Group":                     "income_group",
        "Anchor Year":                      "anchor_year",
        "Years Since Survey":               "dt",
        "Observed Log Consumption":         "obs_log",
        "Predicted Log Consumption":        "pred_log",
        "Observed Consumption":             "obs_cons",
        "Predicted Consumption":            "pred_cons",
        "Lower Band Log (5th)":             "mu_q05_log",
        "Upper Band Log (95th)":            "mu_q95_log",
        "Lower Predictive Band Log (5th)":  "y_q05_log",
        "Upper Predictive Band Log (95th)": "y_q95_log",
        "Residual":                         "resid_cons",
        "Log Residual":                     "resid_log",
    }
    use = use.rename(columns={k: v for k, v in rename.items() if k in use.columns})
    return use


def load_use_parameters():
    """Collect USE posterior summaries from all rolling target-year runs."""
    rows = []
    for mode in MODES:
        save_dir = OUTPUTS_DIR / f"{USE_MODEL_NAME}_{mode}"
        if not save_dir.exists():
            continue
        # Match the new filename pattern from conphi_v1_use.py
        for csv_path in save_dir.rglob(
            f"target_year=*/{USE_MODEL_NAME}_posterior_summary_*.csv"
        ):
            year_str = csv_path.parent.name.split("=")[-1]
            try:
                year = int(year_str)
                pf   = pd.read_csv(csv_path, index_col=0)
                for param, row in pf.iterrows():
                    rows.append({
                        "mode":        mode,
                        "pred_type":   "Nowcast" if mode == "nowcast" else "Forecast",
                        "target_year": year,
                        "param":       param,
                        "param_label": PARAM_MAP_USE.get(param, param),
                        "mean":        row.get("mean",    np.nan),
                        "sd":          row.get("sd",      np.nan),
                        "q05":         row.get("hdi_3%",  np.nan),
                        "q95":         row.get("hdi_97%", np.nan),
                    })
            except Exception as exc:
                print(f"    Skipping {csv_path.name}: {exc}")
    params = pd.DataFrame(rows)
    print(f"  Parameter rows loaded: {len(params):,}")
    return params


# ============================================================
# FIGURE BUILDERS
# ============================================================

def _fig_params_over_time(params):
    """β⁺, β⁻, σ, ν trajectories with 90% HDI ribbons."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    key_params     = ["beta0_pos", "beta0_neg", "sigma", "nu"]
    subplot_titles = [PARAM_MAP_USE.get(p, p) for p in key_params]
    fig = make_subplots(rows=2, cols=2, subplot_titles=subplot_titles,
                        vertical_spacing=0.16, horizontal_spacing=0.10)
    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}

    for idx, param in enumerate(key_params):
        row, col = idx // 2 + 1, idx % 2 + 1
        sub = params[params["param"] == param].copy()
        if len(sub) == 0:
            continue
        for pt, grp in sub.groupby("pred_type"):
            grp = grp.sort_values("target_year")
            c   = colours.get(pt, "#333")
            fig.add_trace(go.Scatter(
                x=pd.concat([grp["target_year"], grp["target_year"].iloc[::-1]]),
                y=pd.concat([grp["q95"], grp["q05"].iloc[::-1]]),
                fill="toself", fillcolor=hex_to_rgba(c, 0.15),
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=grp["target_year"], y=grp["mean"],
                mode="lines+markers",
                line=dict(color=c, width=2), marker=dict(size=5),
                name=pt, legendgroup=pt, showlegend=(idx == 0),
                hovertemplate=(
                    f"<b>{PARAM_MAP_USE.get(param, param)}</b><br>"
                    "Year: %{x}<br>Mean: %{y:.4f}<extra></extra>"
                ),
            ), row=row, col=col)

    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Posterior Parameter Trajectories",
        height=560, template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.08),
        margin=dict(t=80, b=60),
    )
    return fig


def _fig_beta_asymmetry(params):
    """β⁺ vs β⁻ on a single panel with unity reference line."""
    import plotly.graph_objects as go

    pos = params[params["param"] == "beta0_pos"].sort_values("target_year")
    neg = params[params["param"] == "beta0_neg"].sort_values("target_year")
    if len(pos) == 0 or len(neg) == 0:
        return None

    fig = go.Figure()
    for label, grp, colour in [
        ("β⁺ Expansion",   pos, "#2ca02c"),
        ("β⁻ Contraction", neg, "#d62728"),
    ]:
        fig.add_trace(go.Scatter(
            x=pd.concat([grp["target_year"], grp["target_year"].iloc[::-1]]),
            y=pd.concat([grp["q95"], grp["q05"].iloc[::-1]]),
            fill="toself", fillcolor=hex_to_rgba(colour, 0.15),
            line=dict(width=0), showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=grp["target_year"], y=grp["mean"],
            mode="lines+markers",
            line=dict(color=colour, width=2.5), marker=dict(size=6),
            name=label,
            hovertemplate="Year: %{x}<br>Mean: %{y:.4f}<extra></extra>",
        ))

    fig.add_hline(y=1.0, line=dict(dash="dash", color="grey", width=1.2),
                  annotation_text="Full passthrough (β=1)",
                  annotation_position="top right")
    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Asymmetric Passthrough: β⁺ vs β⁻",
        xaxis_title="Target Year", yaxis_title="Passthrough Coefficient",
        height=420, template="plotly_white",
        legend=dict(orientation="h", x=0.25, y=-0.15),
    )
    return fig


def _fig_coverage_calibration(use_df):
    """Coverage calibration curve (nominal vs empirical)."""
    import plotly.graph_objects as go

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["obs_log", "pred_log", "mu_q05_log", "mu_q95_log",
                "y_q05_log", "y_q95_log"]
    ).copy()
    if len(ev) == 0:
        return None, None

    nominal_levels = np.arange(0.05, 1.00, 0.05)
    rows = []
    for band_type, lo_col, hi_col in [
        ("Posterior Mean (μ) 90% CI", "mu_q05_log", "mu_q95_log"),
        ("Predictive (ỹ) 90% CI",    "y_q05_log",  "y_q95_log"),
    ]:
        mu  = ev["pred_log"].values
        lo0 = ev[lo_col].values
        hi0 = ev[hi_col].values
        obs = ev["obs_log"].values
        for nom in nominal_levels:
            scale = nom / 0.90
            lo    = mu + scale * (lo0 - mu)
            hi    = mu + scale * (hi0 - mu)
            cov   = float(np.mean((obs >= lo) & (obs <= hi)))
            rows.append({"band_type": band_type, "nominal": nom, "empirical": cov})

    cal = pd.DataFrame(rows)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines",
        line=dict(dash="dash", color="grey", width=1.5),
        name="Perfect calibration", hoverinfo="skip",
    ))
    colours = {
        "Posterior Mean (μ) 90% CI": "#1f77b4",
        "Predictive (ỹ) 90% CI":    "#ff7f0e",
    }
    for bt, grp in cal.groupby("band_type"):
        grp = grp.sort_values("nominal")
        c   = colours.get(bt, "#333")
        fig.add_trace(go.Scatter(
            x=grp["nominal"], y=grp["empirical"],
            mode="lines+markers",
            line=dict(color=c, width=2), marker=dict(size=6),
            name=bt,
            hovertemplate="Nominal: %{x:.0%}<br>Empirical: %{y:.1%}<extra></extra>",
        ))

    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Coverage Calibration Curve",
        xaxis_title="Nominal Coverage", yaxis_title="Empirical Coverage",
        xaxis=dict(tickformat=".0%"), yaxis=dict(tickformat=".0%"),
        height=440, template="plotly_white",
        legend=dict(x=0.02, y=0.95),
    )
    return fig, cal


def _fig_residuals_by_dt(use_df):
    """Box plots of log residuals by dt (years since survey)."""
    import plotly.graph_objects as go

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "dt"]
    ).copy()
    ev["dt"] = ev["dt"].astype(int)
    if len(ev) == 0:
        return None

    fig = go.Figure()
    for dt in sorted(ev["dt"].unique()):
        sub = ev[ev["dt"] == dt]["resid_log"].values
        fig.add_trace(go.Box(
            y=sub, name=f"dt={dt}",
            marker_color="#1f77b4", line_color="#1f77b4",
            boxmean="sd",
            hovertemplate=f"dt={dt}<br>%{{y:.4f}}<extra></extra>",
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Log Residual Distribution by Years Since Survey (dt)",
        xaxis_title="Years Since Survey",
        yaxis_title="Log Residual (pred − obs)",
        height=440, template="plotly_white", showlegend=False,
    )
    return fig


def _fig_dt_metrics(use_df):
    """R², MAE, MAPE line charts by dt, split by prediction type."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["obs_log", "pred_log", "dt"]
    ).copy()
    ev["dt"] = ev["dt"].astype(int)
    if len(ev) == 0:
        return None, None

    rows_out = []
    for (pt, dt), grp in ev.groupby(["pred_type", "dt"]):
        m = compute_metrics_df(grp)
        if m:
            m.update({"pred_type": pt, "dt": dt})
            rows_out.append(m)
    dt_df = pd.DataFrame(rows_out)
    if len(dt_df) == 0:
        return None, None

    metrics_to_plot = [("r2_log", "R² (Log)"), ("mae_log", "MAE (Log)"), ("mape_pct", "MAPE %")]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[m[1] for m in metrics_to_plot],
                        horizontal_spacing=0.10)

    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}
    for ci, (metric, label) in enumerate(metrics_to_plot):
        for pt, grp in dt_df.groupby("pred_type"):
            grp = grp.sort_values("dt")
            c   = colours.get(pt, "#333")
            fig.add_trace(go.Scatter(
                x=grp["dt"], y=grp[metric],
                mode="lines+markers",
                line=dict(color=c, width=2), marker=dict(size=6),
                name=pt, legendgroup=pt, showlegend=(ci == 0),
                hovertemplate=f"dt=%{{x}}<br>{label}: %{{y:.4f}}<extra></extra>",
            ), row=1, col=ci + 1)

    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Performance Degradation by Years Since Survey",
        height=400, template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.18),
    )
    return fig, dt_df


def _fig_percentile_calibration(use_df):
    """Mean log residual ± SE by percentile."""
    import plotly.graph_objects as go

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "percentile"]
    ).copy()
    if len(ev) == 0:
        return None

    pct_df = (
        ev.groupby(["pred_type", "percentile"])["resid_log"]
        .agg(mean="mean", sd="std", n="count")
        .reset_index()
    )

    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}
    fig = go.Figure()
    for pt, grp in pct_df.groupby("pred_type"):
        grp = grp.sort_values("percentile")
        c   = colours.get(pt, "#333")
        se  = (grp["sd"] / np.sqrt(grp["n"])).values
        fig.add_trace(go.Scatter(
            x=grp["percentile"], y=grp["mean"],
            error_y=dict(type="data", array=se, color=c, thickness=1.5, width=4),
            mode="lines+markers",
            line=dict(color=c, width=2), marker=dict(size=6),
            name=pt,
            hovertemplate="p%{x}<br>Mean bias: %{y:.4f}<extra></extra>",
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Mean Log Residual by Percentile (± 1 SE)",
        xaxis_title="Percentile",
        yaxis_title="Mean Log Residual (pred − obs)",
        height=420, template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.15),
    )
    return fig


def _fig_residuals_by_income_dt(use_df):
    """Mean log residual by income group across dt."""
    import plotly.graph_objects as go

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "dt", "income_group"]
    ).copy()
    ev["dt"] = ev["dt"].astype(int)
    if len(ev) == 0:
        return None

    summary = (
        ev.groupby(["income_group", "dt"])["resid_log"]
        .agg(mean="mean", sd="std", n="count")
        .reset_index()
    )

    fig = go.Figure()
    for ig in INCOME_ORDER:
        sub = summary[summary["income_group"] == ig].sort_values("dt")
        if len(sub) == 0:
            continue
        c  = INCOME_PALETTE.get(ig, "#999")
        se = (sub["sd"] / np.sqrt(sub["n"])).values
        fig.add_trace(go.Scatter(
            x=sub["dt"], y=sub["mean"],
            error_y=dict(type="data", array=se, color=c, thickness=1.5, width=4),
            mode="lines+markers",
            line=dict(color=c, width=2), marker=dict(size=6),
            name=ig,
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="grey", width=1))
    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Mean Log Residual by Income Group × dt",
        xaxis_title="Years Since Survey (dt)",
        yaxis_title="Mean Log Residual",
        height=440, template="plotly_white",
        legend=dict(title="Income Group"),
    )
    return fig


def _fig_residuals_by_region(use_df):
    """Violin plots of log residuals by WB region."""
    import plotly.graph_objects as go

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "region"]
    ).copy()
    if len(ev) == 0:
        return None

    fig = go.Figure()
    for reg in sorted(ev["region"].unique()):
        sub = ev[ev["region"] == reg]["resid_log"].values
        c   = WB_PALETTE.get(reg, "#999")
        fig.add_trace(go.Violin(
            y=sub, name=reg,
            box_visible=True, meanline_visible=True,
            fillcolor=hex_to_rgba(c, 0.50), line_color=c,
            hoverinfo="name+y",
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Log Residual Distribution by World Bank Region",
        yaxis_title="Log Residual (pred − obs)",
        height=480, template="plotly_white", showlegend=False,
    )
    return fig


def _fig_country_mae(use_df):
    """Horizontal bar chart: top 40 countries by mean log MAE."""
    import plotly.graph_objects as go

    ev = use_df[use_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "iso"]
    ).copy()
    if len(ev) == 0:
        return None, None

    country_df = (
        ev.groupby(["iso", "country_name", "region", "income_group"])
        .agg(
            mae_log  = ("resid_log", lambda x: np.mean(np.abs(x))),
            bias_log = ("resid_log", "mean"),
            n_obs    = ("resid_log", "count"),
        )
        .reset_index()
        .sort_values("mae_log", ascending=False)
    )

    top40   = country_df.head(40)
    colours = top40["region"].map(WB_PALETTE).fillna("#999").tolist()

    fig = go.Figure(go.Bar(
        x=top40["mae_log"],
        y=top40["country_name"],
        orientation="h",
        marker_color=colours,
        customdata=np.stack([
            top40["n_obs"].values,
            top40["region"].fillna("Unknown").values,
            top40["income_group"].fillna("Unknown").values,
            top40["bias_log"].round(4).values,
        ], axis=1),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "MAE (log): %{x:.4f}<br>"
            "Bias (log): %{customdata[3]}<br>"
            "n obs: %{customdata[0]}<br>"
            "Region: %{customdata[1]}<br>"
            "Income: %{customdata[2]}<extra></extra>"
        ),
    ))

    fig.update_layout(
        title_text=f"Con φ {VERSION} USE — Top 40 Countries by Log MAE (Validated)",
        xaxis_title="MAE (Log Consumption)",
        yaxis=dict(autorange="reversed"),
        height=900, template="plotly_white",
        margin=dict(l=180),
    )
    return fig, country_df


# ============================================================
# OVERALL SUMMARY TABLE
# ============================================================
def _overall_summary_html(use_df):
    rows_out = []
    for pt, grp in use_df[use_df["data_type"] == "Validated"].groupby("pred_type"):
        m = compute_metrics_df(grp)
        if m:
            m["Prediction Type"] = pt
            rows_out.append(m)
    if not rows_out:
        return "<p>No validated rows found.</p>", pd.DataFrame()
    tbl   = pd.DataFrame(rows_out)
    order = [
        "Prediction Type", "n_obs", "n_countries",
        "r2_log", "mae_log", "rmse_log", "bias_log",
        "r2_cons", "mae_cons", "mape_pct", "mahler",
    ]
    tbl = tbl[[c for c in order if c in tbl.columns]]
    return (
        tbl.to_html(index=False, float_format="{:.4f}".format,
                    border=0, classes="summary-table", justify="center"),
        tbl,
    )


# ============================================================
# HTML ASSEMBLY
# ============================================================
CSS = """
<style>
  body { font-family: "Segoe UI", Arial, sans-serif;
         margin: 32px 48px; background: #fafafa; color: #222; }
  h1   { color: #003366; border-bottom: 3px solid #003366; padding-bottom: 8px; }
  h2   { color: #005b96; margin-top: 40px; border-left: 4px solid #005b96;
         padding-left: 10px; }
  p.desc { color: #555; max-width: 840px; line-height: 1.55; }
  .summary-table { border-collapse: collapse; width: 100%; margin: 16px 0; }
  .summary-table th { background: #003366; color: white; padding: 8px 12px; }
  .summary-table td { padding: 6px 12px; border-bottom: 1px solid #ddd; }
  .summary-table tr:nth-child(even) { background: #eef3f9; }
  .plot-wrap { background: white; border: 1px solid #dde3ea; border-radius: 6px;
               padding: 8px; margin: 20px 0;
               box-shadow: 0 1px 4px rgba(0,0,0,.06); }
  .toc  { background: #e8f0fb; border-radius: 6px; padding: 16px 24px;
          margin-bottom: 32px; display: inline-block; }
  .toc a { display: block; color: #003366; margin: 4px 0; text-decoration: none; }
  .toc a:hover { text-decoration: underline; }
</style>
"""

SECTION_META = [
    ("overall_metrics",        "Overall Performance Metrics"),
    ("parameter_trajectories", "Posterior Parameter Trajectories"),
    ("passthrough_asymmetry",  "Asymmetric Passthrough β⁺ vs β⁻"),
    ("coverage_calibration",   "Coverage Calibration"),
    ("residuals_by_dt",        "Residuals by dt"),
    ("dt_performance",         "Performance Degradation by dt"),
    ("percentile_calibration", "Per-Percentile Bias"),
    ("residuals_by_income",    "Residuals by Income Group × dt"),
    ("residuals_by_region",    "Residuals by World Bank Region"),
    ("country_mae",            "Country-Level MAE"),
]


def _wrap(fig):
    import plotly.io as pio
    return pio.to_html(fig, full_html=False, include_plotlyjs=False)


def _section(anchor, title, desc, content_html):
    return (
        anchor,
        f'<div id="{anchor}" class="plot-wrap">'
        f"<h2>{title}</h2>"
        f'<p class="desc">{desc}</p>'
        f"{content_html}</div>",
    )


def build_report(sections, out_path):
    toc  = "\n".join(
        f'<a href="#{aid}">{label}</a>'
        for aid, label in SECTION_META
        if aid in {s[0] for s in sections}
    )
    body = "\n".join(html for _, html in sections)
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Con φ {VERSION} — USE Diagnostics</title>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
{CSS}
</head>
<body>
<h1>Con φ {VERSION} — USE (Update Survey Estimate) Diagnostics</h1>
<p class="desc">
  Interactive diagnostic report for the Con φ USE sub-model. USE projects
  consumption distributions forward from the most recent survey anchor using
  an asymmetric Student-t GDP passthrough framework with separate expansion
  and contraction passthrough rates. All metrics are computed on
  <strong>validated rows only</strong> (rows with matched observed survey data).
</p>
<div class="toc"><strong>Contents</strong><br>{toc}</div>
{body}
</body></html>"""
    out_path.write_text(html, encoding="utf-8")
    print(f"  Report written → {out_path}")


# ============================================================
# MAIN
# ============================================================
def main():
    try:
        import plotly.io  # noqa
    except ImportError:
        raise ImportError("plotly is required: pip install plotly")

    print("=" * 70)
    print(f"Con φ {VERSION}  ~  USE DIAGNOSTIC REPORT")
    print(f"Output directory: {OUT_DIR}")
    print("=" * 70)

    print("\nLoading data...")
    use_df = load_use_fact()
    params = load_use_parameters()

    # ── Parquet outputs ────────────────────────────────────────
    print("\nWriting parquet tables...")
    if len(params) > 0:
        params.to_parquet(OUT_DIR / "use_diag_params_over_time.parquet", index=False)
        print(f"  use_diag_params_over_time.parquet : {len(params):,} rows")

    keep   = [c for c in [
        "iso", "country_name", "target_year", "pred_year", "percentile",
        "horizon", "pred_type", "region", "sub_region", "income_group",
        "anchor_year", "dt", "obs_log", "pred_log", "resid_log",
        "obs_cons", "pred_cons", "resid_cons",
    ] if c in use_df.columns]
    ev_df  = use_df[use_df["data_type"] == "Validated"][keep].copy()
    ev_df.to_parquet(OUT_DIR / "use_diag_residuals.parquet", index=False)
    print(f"  use_diag_residuals.parquet         : {len(ev_df):,} rows")

    # ── Build sections ─────────────────────────────────────────
    print("\nBuilding figures...")
    sections = []

    # 0. Overall metrics
    tbl_html, tbl_df = _overall_summary_html(use_df)
    if not tbl_df.empty:
        tbl_df.to_parquet(OUT_DIR / "use_diag_overall_metrics.parquet", index=False)
    sections.append(_section(
        "overall_metrics", "Overall Performance Metrics",
        "Top-line metrics for all validated USE predictions, split by Nowcast and Forecast.",
        tbl_html,
    ))

    # 1. Parameter trajectories
    if len(params) > 0:
        fig = _fig_params_over_time(params)
        sections.append(_section(
            "parameter_trajectories", "Posterior Parameter Trajectories",
            "Key posterior means and 90% HDI intervals across rolling target years. "
            "Stable bands signal good prior calibration; drifting means suggest regime shift.",
            _wrap(fig),
        ))

    # 2. Asymmetric passthrough
    if len(params) > 0:
        fig = _fig_beta_asymmetry(params)
        if fig:
            sections.append(_section(
                "passthrough_asymmetry", "Asymmetric GDP Passthrough: β⁺ vs β⁻",
                "Contraction passthrough (β⁻) is empirically larger than expansion "
                "passthrough (β⁺), confirming that income losses translate more strongly "
                "into reduced consumption than equivalent GDP gains do during expansions. "
                "The dashed unity line marks full passthrough.",
                _wrap(fig),
            ))

    # 3. Coverage calibration
    result = _fig_coverage_calibration(use_df)
    if result and result[0] is not None:
        fig, cal_df = result
        cal_df.to_parquet(OUT_DIR / "use_diag_coverage.parquet", index=False)
        sections.append(_section(
            "coverage_calibration", "Coverage Calibration Curve",
            "Empirical vs nominal coverage for posterior mean and predictive intervals. "
            "Points above the diagonal = over-coverage; below = under-coverage.",
            _wrap(fig),
        ))

    # 4. Residuals by dt
    fig = _fig_residuals_by_dt(use_df)
    if fig:
        sections.append(_section(
            "residuals_by_dt", "Log Residual Distribution by Years Since Survey (dt)",
            "Box plots of log residuals as dt increases. Widening spreads confirm expected "
            "uncertainty accumulation; drifting medians indicate systematic passthrough bias.",
            _wrap(fig),
        ))

    # 5. dt performance metrics
    result = _fig_dt_metrics(use_df)
    if result and result[0] is not None:
        fig, dt_df = result
        dt_df.to_parquet(OUT_DIR / "use_diag_dt_metrics.parquet", index=False)
        sections.append(_section(
            "dt_performance", "Performance Degradation by Years Since Survey",
            "R², MAE, and MAPE by dt. The expected degradation curve quantifies how much "
            "predictive accuracy decays as the survey anchor ages.",
            _wrap(fig),
        ))

    # 6. Percentile calibration
    fig = _fig_percentile_calibration(use_df)
    if fig:
        sections.append(_section(
            "percentile_calibration", "Per-Percentile Bias",
            "Mean log residual ± 1 SE at each percentile. Systematic bias at the tails "
            "indicates the distributional tilt parameters (β_p⁺, β_p⁻) may need recalibration.",
            _wrap(fig),
        ))

    # 7. Residuals by income × dt
    fig = _fig_residuals_by_income_dt(use_df)
    if fig:
        sections.append(_section(
            "residuals_by_income", "Mean Log Residual by Income Group × dt",
            "Mean ± 1 SE of log residuals per income group across dt. Persistent positive "
            "bias in low-income countries suggests GDP passthrough is underestimated for "
            "structurally volatile economies.",
            _wrap(fig),
        ))

    # 8. Residuals by region
    fig = _fig_residuals_by_region(use_df)
    if fig:
        reg_df = (
            use_df[use_df["data_type"] == "Validated"]
            .dropna(subset=["resid_log", "region"])
            .groupby(["pred_type", "region"])
            .apply(lambda g: pd.Series(compute_metrics_df(g)))
            .reset_index()
        )
        if len(reg_df) > 0:
            reg_df.to_parquet(OUT_DIR / "use_diag_region_metrics.parquet", index=False)
        sections.append(_section(
            "residuals_by_region", "Log Residual Distribution by World Bank Region",
            "Violin plots reveal structural residual patterns by region. Tall violins "
            "indicate high within-region heterogeneity; shifted centres indicate systematic bias.",
            _wrap(fig),
        ))

    # 9. Country MAE
    result = _fig_country_mae(use_df)
    if result and result[0] is not None:
        fig, country_df = result
        country_df.to_parquet(OUT_DIR / "use_diag_country_mae.parquet", index=False)
        sections.append(_section(
            "country_mae", "Top 40 Countries by Log MAE (Validated)",
            "Countries ranked by mean absolute log-consumption error on validated rows. "
            "Colour encodes World Bank region. These structurally difficult countries "
            "drive the majority of aggregate RMSE.",
            _wrap(fig),
        ))

    # ── Write report ───────────────────────────────────────────
    print("\nAssembling HTML report...")
    build_report(sections, OUT_DIR / "use_diagnostics.html")

    n_parquet = len(list(OUT_DIR.glob("*.parquet")))
    print(f"\nDone — {n_parquet} parquet files + HTML report in:\n  {OUT_DIR}")


if __name__ == "__main__":
    main()

Con φ v1  ~  USE DIAGNOSTIC REPORT
Output directory: /content/drive/MyDrive/conphi/outputs/conphi_v1_report/diagnostics_use

Loading data...
  USE rows from fact table: 253,539
  Parameter rows loaded: 66

Writing parquet tables...
  use_diag_params_over_time.parquet : 66 rows
  use_diag_residuals.parquet         : 40,095 rows

Building figures...

Assembling HTML report...
  Report written → /content/drive/MyDrive/conphi/outputs/conphi_v1_report/diagnostics_use/use_diagnostics.html

Done — 7 parquet files + HTML report in:
  /content/drive/MyDrive/conphi/outputs/conphi_v1_report/diagnostics_use


In [17]:
#!/usr/bin/env python3
"""
Conφ — WASE Model Diagnostic Report
=====================================
Produces a self-contained HTML report (Plotly) and parquet tables for the
WASE (Without Any Survey Estimate / Conφ-Level) model.

OUTPUTS
-------
  powerbi_data/diagnostics_wase/
    wase_diagnostics.html                   — standalone interactive report
    wase_diag_params_forest.parquet         — coefficient posterior summaries
    wase_diag_residuals.parquet             — row-level residuals with covariates
    wase_diag_coverage.parquet              — coverage calibration table
    wase_diag_country_mae.parquet           — country-level MAE summary
    wase_diag_region_metrics.parquet        — metrics by WB region
    wase_diag_income_metrics.parquet        — metrics by income group
    wase_diag_overall_metrics.parquet       — top-line metrics table
    wase_diag_percentile_metrics.parquet    — metrics by percentile
    wase_diag_horizon_metrics.parquet       — metrics by horizon (0–5)

USAGE
-----
Run from the same Colab environment as the main Power BI prep script.
All paths inherit from BASE_DIR.
"""

import numpy as np
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION — mirrors main prep script
# ============================================================

VERSION   = "v1"
BASE_DIR  = Path("/content/drive/MyDrive/conphi")

OUTPUTS_DIR = BASE_DIR / "outputs"
REPORT_DIR  = OUTPUTS_DIR / f"conphi_{VERSION}_report"
OUT_DIR     = REPORT_DIR / "diagnostics_wase"
FACT_PATH   = REPORT_DIR / "fact_predictions.parquet"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODES = [
    "nowcast",
    "forecast_1yr", "forecast_2yr",
    "forecast_3yr", "forecast_4yr", "forecast_5yr",
]

# Human-readable coefficient labels — extend as your WASE spec evolves
PARAM_MAP_WASE = {
    "intercept":             "Intercept (Global Baseline)",
    "gdp_elasticity":        "GDP p.c. Elasticity",
    "gdp_curvature":         "GDP Curvature (Non-linear)",
    "gdp_growth_effect":     "GDP Growth Effect",
    "u5_mortality_effect":   "Under-5 Mortality Effect",
    "rural_share_effect":    "Rural Population Share Effect",
    "female_education_effect": "Female Education Effect",
    "gov_rev_effect":        "Government Revenue Effect",
    "res_rents_effect":      "Resource Rents Effect",
    "gov_exp_effect":        "Government Expenditure Effect",
    "baseline_inequality":   "Baseline Inequality (Shape)",
    "noise_baseline":        "Observation Noise (Baseline)",
    "noise_tail_widening":   "Observation Noise (Tail Widening)",
    # RBF / spline components
    "rbf_weight_1":          "RBF Spline Weight 1",
    "rbf_weight_2":          "RBF Spline Weight 2",
    "rbf_weight_3":          "RBF Spline Weight 3",
}

INCOME_ORDER = [
    "Low income", "Lower middle income",
    "Upper middle income", "High income", "Unknown",
]

WB_PALETTE = {
    "East Asia & Pacific":        "#1f77b4",
    "Europe & Central Asia":      "#aec7e8",
    "Latin America & Caribbean":  "#ff7f0e",
    "Middle East & North Africa": "#ffbb78",
    "South Asia":                 "#2ca02c",
    "Sub-Saharan Africa":         "#d62728",
    "Unknown":                    "#999999",
}

INCOME_PALETTE = {
    "Low income":          "#d62728",
    "Lower middle income": "#ff7f0e",
    "Upper middle income": "#2ca02c",
    "High income":         "#1f77b4",
    "Unknown":             "#999999",
}

# ============================================================
# HELPERS
# ============================================================

LOG_CLIP = (-20.0, 20.0)

def hex_to_rgba(hex_color: str, alpha: float) -> str:
    """Convert a 6-digit hex colour to an rgba() string."""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def safe_exp(x):
    return np.exp(np.clip(x, *LOG_CLIP))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return float(1.0 - ss_res / ss_tot) if ss_tot > 1e-8 else np.nan


def compute_metrics_df(df):
    """Return a dict of scalar metrics; df must have obs_log, pred_log."""
    ev = df.dropna(subset=["obs_log", "pred_log"]).copy()
    if len(ev) == 0:
        return {}
    obs_l   = ev["obs_log"].values
    pred_l  = ev["pred_log"].values
    obs_c   = safe_exp(obs_l)
    pred_c  = safe_exp(pred_l)
    resid_l = pred_l - obs_l
    resid_c = pred_c - obs_c
    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.where(obs_c > 0, np.abs(resid_c / obs_c), np.nan)
    iso_col = "iso" if "iso" in ev.columns else None
    return {
        "n_obs":       int(len(ev)),
        "n_countries": int(ev[iso_col].nunique()) if iso_col else np.nan,
        "mae_log":     float(np.mean(np.abs(resid_l))),
        "rmse_log":    float(np.sqrt(np.mean(resid_l ** 2))),
        "bias_log":    float(resid_l.mean()),
        "r2_log":      r2_score(obs_l, pred_l),
        "mae_cons":    float(np.mean(np.abs(resid_c))),
        "rmse_cons":   float(np.sqrt(np.mean(resid_c ** 2))),
        "bias_cons":   float(resid_c.mean()),
        "r2_cons":     r2_score(obs_c, pred_c),
        "mape_pct":    float(np.nanmean(ape) * 100),
        "mahler":      float(
            ev.assign(_ae=np.abs(resid_l))
              .groupby(iso_col)["_ae"].mean().mean() * 100
        ) if iso_col else np.nan,
    }


# ============================================================
# DATA LOADING
# ============================================================
def load_wase_fact():
    """Pull WASE rows from the unified fact table."""
    if not FACT_PATH.exists():
        raise FileNotFoundError(
            f"Fact table not found at {FACT_PATH}.\n"
            "Run the main Power BI prep script first."
        )
    df   = pd.read_parquet(FACT_PATH)
    wase = df[df["Model Type"] == "WASE"].copy()
    print(f"  WASE rows from fact table: {len(wase):,}")

    rename = {
        "Country Code":                    "iso",
        "Country":                         "country_name",
        "Year":                            "focal_year",
        "Prediction Year":                 "pred_year",
        "Percentile":                      "percentile",
        "Horizon":                         "horizon",
        "Prediction Type":                 "pred_type",
        "Data Type":                       "data_type",
        "Region":                          "region",
        "Sub-Region":                      "sub_region",
        "Income Group":                    "income_group",
        "Observed Log Consumption":        "obs_log",
        "Predicted Log Consumption":       "pred_log",
        "Observed Consumption":            "obs_cons",
        "Predicted Consumption":           "pred_cons",
        "Lower Band Log (5th)":            "mu_q05_log",
        "Upper Band Log (95th)":           "mu_q95_log",
        "Lower Predictive Band Log (5th)": "y_q05_log",
        "Upper Predictive Band Log (95th)":"y_q95_log",
        "Residual":                        "resid_cons",
        "Log Residual":                    "resid_log",
    }
    wase = wase.rename(columns={k: v for k, v in rename.items() if k in wase.columns})
    return wase


def load_wase_parameters():
    """
    Load per-fold posterior summaries from rolling_params_master files.
    Falls back gracefully if files are missing.
    """
    rows = []
    for mode in MODES:
        master_dir = OUTPUTS_DIR / f"conphi_{VERSION}_wase_{mode}"  / "master"
        path = master_dir / f"conphi_{VERSION}_wase_rolling_params_master_{mode}.parquet"
        if not path.exists():
            continue
        pf = pd.read_parquet(path)
        for _, row in pf.iterrows():
            param = str(row.get("parameter", ""))
            rows.append({
                "mode":        mode,
                "pred_type":   "Nowcast" if mode == "nowcast" else "Forecast",
                "focal_year":  int(row.get("focal_year", 0)),
                "param":       param,
                "param_label": PARAM_MAP_WASE.get(param, param),
                "mean":        row.get("posterior_mean", np.nan),
                "sd":          row.get("posterior_sd",   np.nan),
                "q05":         row.get("posterior_q05",  np.nan),
                "q95":         row.get("posterior_q95",  np.nan),
            })
    params = pd.DataFrame(rows)
    print(f"  WASE parameter rows loaded: {len(params):,}")
    return params


# ============================================================
# FIGURE BUILDERS
# ============================================================

def _fig_forest_plot(params):
    """
    Coefficient forest plot: one dot per parameter with 90 % HDI error bars,
    averaged across folds for each prediction type.
    Only shows parameters present in PARAM_MAP_WASE.
    """
    import plotly.graph_objects as go

    mapped = params[params["param"].isin(PARAM_MAP_WASE)].copy()
    if len(mapped) == 0:
        return None

    summary = (
        mapped.groupby(["pred_type", "param", "param_label"])
        .agg(mean=("mean", "mean"), q05=("q05", "mean"), q95=("q95", "mean"))
        .reset_index()
        .sort_values(["pred_type", "mean"], ascending=[True, False])
    )

    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}
    fig = go.Figure()

    for pt in summary["pred_type"].unique():
        sub = summary[summary["pred_type"] == pt].reset_index(drop=True)
        c   = colours.get(pt, "#333")
        fig.add_trace(go.Scatter(
            x=sub["mean"],
            y=sub["param_label"],
            error_x=dict(
                type="data",
                symmetric=False,
                array=(sub["q95"] - sub["mean"]).clip(lower=0).values,
                arrayminus=(sub["mean"] - sub["q05"]).clip(lower=0).values,
                color=c, thickness=2, width=6,
            ),
            mode="markers",
            marker=dict(size=9, color=c, symbol="circle"),
            name=pt,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Mean: %{x:.4f}<br>"
                f"Pred type: {pt}<extra></extra>"
            ),
        ))

    fig.add_vline(x=0, line=dict(dash="dash", color="grey", width=1.2))
    fig.update_layout(
        title_text="WASE — Coefficient Forest Plot (Posterior Means ± 90 % HDI, Avg Across Folds)",
        xaxis_title="Posterior Mean",
        yaxis=dict(autorange="reversed"),
        height=max(500, len(summary["param_label"].unique()) * 45),
        template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.10),
        margin=dict(l=260),
    )
    return fig


def _fig_coeff_trajectories(params, param_keys=None):
    """
    Line charts showing how key coefficients evolve across focal years.
    param_keys: list of raw param names to plot; defaults to top structural ones.
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    if param_keys is None:
        param_keys = [
            "gdp_elasticity", "u5_mortality_effect",
            "rural_share_effect", "gov_rev_effect",
            "res_rents_effect", "gdp_growth_effect",
        ]
    # Keep only those that exist in the data
    available = [p for p in param_keys if p in params["param"].values]
    if not available:
        return None

    nc = 2
    nr = int(np.ceil(len(available) / nc))
    subplot_titles = [PARAM_MAP_WASE.get(p, p) for p in available]
    fig = make_subplots(rows=nr, cols=nc, subplot_titles=subplot_titles,
                        vertical_spacing=0.14, horizontal_spacing=0.10)

    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}

    for idx, param in enumerate(available):
        row, col = idx // nc + 1, idx % nc + 1
        sub = params[params["param"] == param].sort_values("focal_year")
        if len(sub) == 0:
            continue
        for pt, grp in sub.groupby("pred_type"):
            grp = grp.sort_values("focal_year")
            c   = colours.get(pt, "#333")
            # HDI ribbon
            fig.add_trace(go.Scatter(
                x=pd.concat([grp["focal_year"], grp["focal_year"].iloc[::-1]]),
                y=pd.concat([grp["q95"], grp["q05"].iloc[::-1]]),
                fill="toself", fillcolor=hex_to_rgba(c, 0.15),
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ), row=row, col=col)
            fig.add_trace(go.Scatter(
                x=grp["focal_year"], y=grp["mean"],
                mode="lines+markers",
                line=dict(color=c, width=2), marker=dict(size=5),
                name=pt, legendgroup=pt, showlegend=(idx == 0),
                hovertemplate=(
                    f"<b>{PARAM_MAP_WASE.get(param, param)}</b><br>"
                    "Year: %{x}<br>Mean: %{y:.4f}<extra></extra>"
                ),
            ), row=row, col=col)

    fig.update_layout(
        title_text="WASE — Coefficient Stability Across Rolling Folds",
        height=nr * 260, template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.06),
        margin=dict(t=80, b=60),
    )
    return fig


def _fig_predictor_importance(params):
    """
    Predictor importance: |mean posterior| / posterior_sd (signal-to-noise ratio)
    averaged across folds — a simple proxy for practical importance.
    """
    import plotly.graph_objects as go

    mapped = params[params["param"].isin(PARAM_MAP_WASE)].copy()
    if len(mapped) == 0:
        return None

    imp = (
        mapped.groupby(["pred_type", "param"])
        .apply(lambda g: pd.Series({
            "snr":      (np.abs(g["mean"]) / g["sd"].replace(0, np.nan)).mean(),
            "abs_mean": np.abs(g["mean"]).mean(),
            "mean":     g["mean"].mean(),
        }))
        .reset_index()
    )
    imp["param_label"] = imp["param"].map(PARAM_MAP_WASE).fillna(imp["param"])
    imp = imp.sort_values("snr", ascending=True)

    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}
    fig = go.Figure()

    for pt in imp["pred_type"].unique():
        sub = imp[imp["pred_type"] == pt]
        c   = colours.get(pt, "#333")
        fig.add_trace(go.Bar(
            x=sub["snr"],
            y=sub["param_label"],
            orientation="h",
            name=pt,
            marker_color=hex_to_rgba(c, 0.80),
            hovertemplate=(
                "<b>%{y}</b><br>"
                f"Pred type: {pt}<br>"
                "Signal-to-noise: %{x:.2f}<extra></extra>"
            ),
        ))

    fig.add_vline(x=1, line=dict(dash="dot", color="grey", width=1.2),
                  annotation_text="SNR = 1", annotation_position="top right")
    fig.update_layout(
        title_text="WASE — Predictor Importance (|Mean| / SD, Averaged Across Folds)",
        xaxis_title="|Posterior Mean| / Posterior SD (Signal-to-Noise)",
        yaxis=dict(autorange="reversed"),
        height=max(450, len(imp["param_label"].unique()) * 42),
        barmode="group",
        template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.10),
        margin=dict(l=260),
    )
    return fig


def _fig_coverage_calibration(wase_df):
    """Rescaled empirical coverage from the available 90 % CI bands."""
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["obs_log", "pred_log", "mu_q05_log", "mu_q95_log",
                "y_q05_log", "y_q95_log"]
    ).copy()
    if len(ev) == 0:
        return None, None

    nominal_levels = np.arange(0.05, 1.00, 0.05)
    rows = []
    for band_type, lo_col, hi_col in [
        ("Posterior Mean (μ) 90% CI", "mu_q05_log", "mu_q95_log"),
        ("Predictive (ỹ) 90% CI",    "y_q05_log",  "y_q95_log"),
    ]:
        mu  = ev["pred_log"].values
        lo0 = ev[lo_col].values
        hi0 = ev[hi_col].values
        obs = ev["obs_log"].values
        for nom in nominal_levels:
            scale = nom / 0.90
            lo    = mu + scale * (lo0 - mu)
            hi    = mu + scale * (hi0 - mu)
            cov   = float(np.mean((obs >= lo) & (obs <= hi)))
            rows.append({"band_type": band_type, "nominal": nom, "empirical": cov})

    cal = pd.DataFrame(rows)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines",
        line=dict(dash="dash", color="grey", width=1.5),
        name="Perfect calibration", hoverinfo="skip",
    ))
    colours = {
        "Posterior Mean (μ) 90% CI": "#1f77b4",
        "Predictive (ỹ) 90% CI":    "#ff7f0e",
    }
    for bt, grp in cal.groupby("band_type"):
        grp = grp.sort_values("nominal")
        c   = colours.get(bt, "#333")
        fig.add_trace(go.Scatter(
            x=grp["nominal"], y=grp["empirical"],
            mode="lines+markers",
            line=dict(color=c, width=2), marker=dict(size=6),
            name=bt,
            hovertemplate="Nominal: %{x:.0%}<br>Empirical: %{y:.1%}<extra></extra>",
        ))

    fig.update_layout(
        title_text="WASE — Coverage Calibration Curve",
        xaxis_title="Nominal Coverage",
        yaxis_title="Empirical Coverage",
        xaxis=dict(tickformat=".0%"), yaxis=dict(tickformat=".0%"),
        height=440, template="plotly_white",
        legend=dict(x=0.02, y=0.95),
    )
    return fig, cal


def _fig_residuals_by_region(wase_df):
    """Violin plots of log residuals by WB region."""
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "region"]
    ).copy()
    if len(ev) == 0:
        return None

    fig = go.Figure()
    for reg in sorted(ev["region"].unique()):
        sub = ev[ev["region"] == reg]["resid_log"].values
        c   = WB_PALETTE.get(reg, "#999")
        fig.add_trace(go.Violin(
            y=sub, name=reg,
            box_visible=True, meanline_visible=True,
            fillcolor=hex_to_rgba(c, 0.50), line_color=c,
            hoverinfo="name+y",
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    fig.update_layout(
        title_text="WASE — Log Residual Distribution by World Bank Region",
        yaxis_title="Log Residual (pred − obs)",
        height=480, template="plotly_white", showlegend=False,
    )
    return fig


def _fig_residuals_by_income(wase_df):
    """Box plots of log residuals by income group."""
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "income_group"]
    ).copy()
    if len(ev) == 0:
        return None

    fig = go.Figure()
    for ig in INCOME_ORDER:
        sub = ev[ev["income_group"] == ig]["resid_log"].values
        if len(sub) == 0:
            continue
        c = INCOME_PALETTE.get(ig, "#999")
        fig.add_trace(go.Box(
            y=sub, name=ig,
            marker_color=c, line_color=c,
            boxmean="sd",
            hovertemplate=f"{ig}<br>%{{y:.4f}}<extra></extra>",
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    fig.update_layout(
        title_text="WASE — Log Residual Distribution by Income Group",
        yaxis_title="Log Residual (pred − obs)",
        height=460, template="plotly_white", showlegend=False,
    )
    return fig


def _fig_percentile_calibration(wase_df):
    """Mean log residual ± 1 SE at each percentile — distributional bias check."""
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "percentile"]
    ).copy()
    if len(ev) == 0:
        return None

    pct_df = (
        ev.groupby(["pred_type", "percentile"])["resid_log"]
        .agg(mean="mean", sd="std", n="count")
        .reset_index()
    )

    colours = {"Nowcast": "#1f77b4", "Forecast": "#ff7f0e"}
    fig = go.Figure()
    for pt, grp in pct_df.groupby("pred_type"):
        grp = grp.sort_values("percentile")
        c   = colours.get(pt, "#333")
        se  = (grp["sd"] / np.sqrt(grp["n"])).values
        fig.add_trace(go.Scatter(
            x=grp["percentile"], y=grp["mean"],
            error_y=dict(type="data", array=se, color=c, thickness=1.5, width=4),
            mode="lines+markers",
            line=dict(color=c, width=2), marker=dict(size=6),
            name=pt,
            hovertemplate="p%{x}<br>Mean bias: %{y:.4f}<extra></extra>",
        ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    fig.update_layout(
        title_text="WASE — Mean Log Residual by Consumption Percentile (± 1 SE)",
        xaxis_title="Percentile",
        yaxis_title="Mean Log Residual (pred − obs)",
        height=420, template="plotly_white",
        legend=dict(orientation="h", x=0.3, y=-0.15),
    )
    return fig


def _fig_horizon_metrics(wase_df):
    """
    For WASE forecast rows: R² and MAE by horizon (0–5 years).
    Horizon 0 = nowcast; 1–5 = forecast. Shows how far ahead
    the structural model holds up without any survey anchor.
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["obs_log", "pred_log", "horizon"]
    ).copy()
    ev["horizon"] = ev["horizon"].astype(int)
    if len(ev) == 0:
        return None, None

    rows_out = []
    for h, grp in ev.groupby("horizon"):
        m = compute_metrics_df(grp)
        if m:
            m["horizon"] = h
            rows_out.append(m)
    h_df = pd.DataFrame(rows_out).sort_values("horizon")
    if len(h_df) == 0:
        return None, None

    metrics = [("r2_log", "R² (Log)"), ("mae_log", "MAE (Log)"), ("mape_pct", "MAPE %")]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[m[1] for m in metrics],
                        horizontal_spacing=0.10)

    for ci, (metric, label) in enumerate(metrics):
        fig.add_trace(go.Scatter(
            x=h_df["horizon"], y=h_df[metric],
            mode="lines+markers",
            line=dict(color="#005b96", width=2.5),
            marker=dict(size=8, color="#005b96"),
            hovertemplate=f"h=%{{x}}<br>{label}: %{{y:.4f}}<extra></extra>",
            showlegend=False,
        ), row=1, col=ci + 1)

    fig.update_layout(
        title_text="WASE — Performance by Forecast Horizon (0 = Nowcast)",
        height=380, template="plotly_white",
    )
    return fig, h_df


def _fig_country_mae(wase_df):
    """Horizontal bar chart: top 40 countries by mean log MAE."""
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "iso"]
    ).copy()
    if len(ev) == 0:
        return None, None

    country_df = (
        ev.groupby(["iso", "country_name", "region", "income_group"])
        .agg(
            mae_log  = ("resid_log", lambda x: np.mean(np.abs(x))),
            bias_log = ("resid_log", "mean"),
            n_obs    = ("resid_log", "count"),
        )
        .reset_index()
        .sort_values("mae_log", ascending=False)
    )

    top40   = country_df.head(40)
    colours = top40["region"].map(WB_PALETTE).fillna("#999").tolist()

    fig = go.Figure(go.Bar(
        x=top40["mae_log"],
        y=top40["country_name"],
        orientation="h",
        marker_color=colours,
        customdata=np.stack([
            top40["n_obs"].values,
            top40["region"].fillna("Unknown").values,
            top40["income_group"].fillna("Unknown").values,
            top40["bias_log"].round(4).values,
        ], axis=1),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "MAE (log): %{x:.4f}<br>"
            "Bias (log): %{customdata[3]}<br>"
            "n obs: %{customdata[0]}<br>"
            "Region: %{customdata[1]}<br>"
            "Income: %{customdata[2]}<extra></extra>"
        ),
    ))

    fig.update_layout(
        title_text="WASE — Top 40 Countries by Log-Consumption MAE (Validated)",
        xaxis_title="MAE (Log Consumption)",
        yaxis=dict(autorange="reversed"),
        height=900, template="plotly_white",
        margin=dict(l=180),
    )
    return fig, country_df


def _fig_location_shape_scatter(wase_df):
    """
    Scatter of observed log consumption (location proxy) vs
    log residual, coloured by percentile.  Reveals whether
    the model systematically over/under-predicts at high or
    low consumption levels (shape mis-specification).
    """
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["obs_log", "resid_log", "percentile"]
    ).copy()
    if len(ev) == 0:
        return None

    # Sample for performance (max 15k points)
    if len(ev) > 15_000:
        ev = ev.sample(15_000, random_state=42)

    pct = ev["percentile"].values
    pct_norm = (pct - pct.min()) / max(pct.max() - pct.min(), 1)

    fig = go.Figure(go.Scattergl(
        x=ev["obs_log"],
        y=ev["resid_log"],
        mode="markers",
        marker=dict(
            size=3,
            color=pct_norm,
            colorscale="RdYlBu_r",
            showscale=True,
            colorbar=dict(
                title="Percentile",
                tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                ticktext=["p5", "p25", "p50", "p75", "p95"],
            ),
            opacity=0.55,
        ),
        hovertemplate=(
            "Obs log: %{x:.3f}<br>"
            "Residual: %{y:.3f}<br>"
            f"Country: {ev['country_name'].values if 'country_name' in ev.columns else ''}"
            "<extra></extra>"
        ),
    ))

    fig.add_hline(y=0, line=dict(dash="dash", color="red", width=1))
    # LOESS-style annotation via a rolling average
    ev_sorted = ev.sort_values("obs_log")
    window = max(1, len(ev_sorted) // 50)
    rolling_mean = ev_sorted["resid_log"].rolling(window, center=True).mean()
    fig.add_trace(go.Scatter(
        x=ev_sorted["obs_log"], y=rolling_mean,
        mode="lines",
        line=dict(color="black", width=2.5, dash="solid"),
        name="Smoothed mean residual",
        hoverinfo="skip",
    ))

    fig.update_layout(
        title_text="WASE — Residual vs Observed Consumption Level (Location–Shape Diagnostic)",
        xaxis_title="Observed Log Consumption",
        yaxis_title="Log Residual (pred − obs)",
        height=520, template="plotly_white",
    )
    return fig


def _fig_income_percentile_heatmap(wase_df):
    """
    Heatmap of mean log residual at each income-group × percentile cell.
    Reveals interaction between country wealth and distributional fit.
    """
    import plotly.graph_objects as go

    ev = wase_df[wase_df["data_type"] == "Validated"].dropna(
        subset=["resid_log", "income_group", "percentile"]
    ).copy()
    if len(ev) == 0:
        return None

    pivot = (
        ev.groupby(["income_group", "percentile"])["resid_log"]
        .mean()
        .unstack("percentile")
    )
    # Reorder rows
    pivot = pivot.reindex([ig for ig in INCOME_ORDER if ig in pivot.index])

    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale="RdBu",
        zmid=0,
        colorbar=dict(title="Mean Log Residual"),
        hovertemplate=(
            "Percentile: %{x}<br>"
            "Income group: %{y}<br>"
            "Mean residual: %{z:.4f}<extra></extra>"
        ),
    ))

    fig.update_layout(
        title_text="WASE — Mean Log Residual Heatmap: Income Group × Percentile",
        xaxis_title="Percentile",
        yaxis_title="Income Group",
        height=360, template="plotly_white",
    )
    return fig


# ============================================================
# SUMMARY TABLE
# ============================================================
def _overall_summary_html(wase_df):
    rows_out = []
    for pt, grp in wase_df[wase_df["data_type"] == "Validated"].groupby("pred_type"):
        m = compute_metrics_df(grp)
        if m:
            m["Prediction Type"] = pt
            rows_out.append(m)
    if not rows_out:
        return "<p>No validated rows found.</p>", pd.DataFrame()
    tbl = pd.DataFrame(rows_out)
    order = [
        "Prediction Type", "n_obs", "n_countries",
        "r2_log", "mae_log", "rmse_log", "bias_log",
        "r2_cons", "mae_cons", "mape_pct", "mahler",
    ]
    tbl = tbl[[c for c in order if c in tbl.columns]]
    return (
        tbl.to_html(index=False, float_format="{:.4f}".format,
                    border=0, classes="summary-table", justify="center"),
        tbl,
    )


# ============================================================
# HTML ASSEMBLY
# ============================================================
CSS = """
<style>
  body { font-family: "Segoe UI", Arial, sans-serif;
         margin: 32px 48px; background: #fafafa; color: #222; }
  h1   { color: #6b2d0e; border-bottom: 3px solid #6b2d0e; padding-bottom: 8px; }
  h2   { color: #a04020; margin-top: 40px; border-left: 4px solid #a04020;
         padding-left: 10px; }
  p.desc { color: #555; max-width: 840px; line-height: 1.55; }
  .summary-table { border-collapse: collapse; width: 100%; margin: 16px 0; }
  .summary-table th { background: #6b2d0e; color: white; padding: 8px 12px; }
  .summary-table td { padding: 6px 12px; border-bottom: 1px solid #ddd; }
  .summary-table tr:nth-child(even) { background: #fdf0eb; }
  .plot-wrap { background: white; border: 1px solid #dde3ea; border-radius: 6px;
               padding: 8px; margin: 20px 0;
               box-shadow: 0 1px 4px rgba(0,0,0,.06); }
  .toc  { background: #fbeee8; border-radius: 6px; padding: 16px 24px;
          margin-bottom: 32px; display: inline-block; }
  .toc a { display: block; color: #6b2d0e; margin: 4px 0; text-decoration: none; }
  .toc a:hover { text-decoration: underline; }
</style>
"""

SECTION_META = [
    ("overall_metrics",         "Overall Performance Metrics"),
    ("forest_plot",             "Coefficient Forest Plot"),
    ("predictor_importance",    "Predictor Importance"),
    ("coefficient_trajectories","Coefficient Stability Across Folds"),
    ("coverage_calibration",    "Coverage Calibration"),
    ("residuals_by_region",     "Residuals by World Bank Region"),
    ("residuals_by_income",     "Residuals by Income Group"),
    ("percentile_calibration",  "Per-Percentile Bias"),
    ("income_percentile_heatmap","Income Group × Percentile Heatmap"),
    ("location_shape_scatter",  "Location–Shape Diagnostic Scatter"),
    ("horizon_metrics",         "Performance by Forecast Horizon"),
    ("country_mae",             "Country-Level MAE"),
]


def _wrap(fig):
    import plotly.io as pio
    return pio.to_html(fig, full_html=False, include_plotlyjs=False)


def _section(anchor, title, desc, content_html):
    return (
        anchor,
        f'<div id="{anchor}" class="plot-wrap">'
        f"<h2>{title}</h2>"
        f'<p class="desc">{desc}</p>'
        f"{content_html}</div>",
    )


def build_report(sections, out_path):
    toc = "\n".join(
        f'<a href="#{aid}">{label}</a>'
        for aid, label in SECTION_META
        if aid in {s[0] for s in sections}
    )
    body = "\n".join(html for _, html in sections)
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Conφ — WASE Diagnostics</title>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
{CSS}
</head>
<body>
<h1>Conφ — WASE (Conφ-Level) Model Diagnostics</h1>
<p class="desc">
  Interactive diagnostic report for the WASE sub-model of the Conφ system.
  WASE predicts full consumption distributions from structural country indicators
  (GDP, mortality, rural share, fiscal variables) using a log-logistic
  location–shape decomposition with hierarchical random effects and RBF splines.
  All metrics are computed on <strong>validated rows only</strong>
  (rows with matched observed survey data).
</p>
<div class="toc"><strong>Contents</strong><br>{toc}</div>
{body}
</body></html>"""
    out_path.write_text(html, encoding="utf-8")
    print(f"  Report written → {out_path}")


# ============================================================
# MAIN
# ============================================================
def main():
    try:
        import plotly.io  # noqa
    except ImportError:
        raise ImportError("plotly is required: pip install plotly")

    print("=" * 70)
    print("Conφ — WASE DIAGNOSTIC REPORT")
    print(f"Output directory: {OUT_DIR}")
    print("=" * 70)

    print("\nLoading data...")
    wase_df = load_wase_fact()
    params  = load_wase_parameters()

    # ── Parquet outputs ────────────────────────────────────────────
    print("\nWriting parquet tables...")
    if len(params) > 0:
        params.to_parquet(OUT_DIR / "wase_diag_params_forest.parquet", index=False)
        print(f"  wase_diag_params_forest.parquet        : {len(params):,} rows")

    keep = [c for c in [
        "iso", "country_name", "focal_year", "pred_year", "percentile",
        "horizon", "pred_type", "region", "sub_region", "income_group",
        "obs_log", "pred_log", "resid_log", "obs_cons", "pred_cons", "resid_cons",
    ] if c in wase_df.columns]
    ev_df = wase_df[wase_df["data_type"] == "Validated"][keep].copy()
    ev_df.to_parquet(OUT_DIR / "wase_diag_residuals.parquet", index=False)
    print(f"  wase_diag_residuals.parquet            : {len(ev_df):,} rows")

    # ── Build sections ─────────────────────────────────────────────
    print("\nBuilding figures...")
    sections = []

    # 0. Overall summary
    tbl_html, tbl_df = _overall_summary_html(wase_df)
    if not tbl_df.empty:
        tbl_df.to_parquet(OUT_DIR / "wase_diag_overall_metrics.parquet", index=False)
    sections.append(_section(
        "overall_metrics", "Overall Performance Metrics",
        "Top-line metrics for all validated WASE predictions, "
        "split by Nowcast and Forecast.",
        tbl_html,
    ))

    # 1. Forest plot
    if len(params) > 0:
        fig = _fig_forest_plot(params)
        if fig:
            sections.append(_section(
                "forest_plot",
                "Coefficient Forest Plot",
                "Posterior means ± 90 % HDI for all structural covariates, "
                "averaged across LOCO folds. Parameters whose HDI straddles "
                "zero have little empirical signal in the data.",
                _wrap(fig),
            ))

    # 2. Predictor importance
    if len(params) > 0:
        fig = _fig_predictor_importance(params)
        if fig:
            sections.append(_section(
                "predictor_importance",
                "Predictor Importance (Signal-to-Noise Ratio)",
                "Ratio of |posterior mean| to posterior SD — a simple proxy for "
                "practical importance. SNR > 1 means the signal exceeds the noise; "
                "SNR < 1 indicates the posterior barely moved from the prior. "
                "Note: this does not correct for collinearity between predictors.",
                _wrap(fig),
            ))

    # 3. Coefficient trajectories
    if len(params) > 0:
        fig = _fig_coeff_trajectories(params)
        if fig:
            sections.append(_section(
                "coefficient_trajectories",
                "Coefficient Stability Across Rolling Folds",
                "How key structural coefficients evolve across focal years. "
                "Stable coefficients confirm parameter identifiability; drifting "
                "ones suggest sensitivity to the leave-one-country-out sample.",
                _wrap(fig),
            ))

    # 4. Coverage calibration
    result = _fig_coverage_calibration(wase_df)
    if result and result[0] is not None:
        fig, cal_df = result
        cal_df.to_parquet(OUT_DIR / "wase_diag_coverage.parquet", index=False)
        sections.append(_section(
            "coverage_calibration",
            "Coverage Calibration Curve",
            "Empirical vs nominal coverage for the posterior mean (epistemic) "
            "and predictive (total) intervals, rescaled from the 90 % band. "
            "Post-hoc CI shrinkage (factor 0.55) has already been applied. "
            "Points above the diagonal = over-coverage; below = under-coverage.",
            _wrap(fig),
        ))

    # 5. Residuals by region (violin)
    fig = _fig_residuals_by_region(wase_df)
    if fig:
        reg_df = (
            wase_df[wase_df["data_type"] == "Validated"]
            .dropna(subset=["resid_log", "region"])
            .groupby(["pred_type", "region"])
            .apply(lambda g: pd.Series(compute_metrics_df(g)))
            .reset_index()
        )
        if len(reg_df) > 0:
            reg_df.to_parquet(OUT_DIR / "wase_diag_region_metrics.parquet", index=False)
        sections.append(_section(
            "residuals_by_region",
            "Log Residual Distribution by World Bank Region",
            "Violin plots showing within-region heterogeneity of log residuals. "
            "Tall violins indicate high within-region variability; "
            "shifted centres indicate systematic regional misfit that the "
            "structural covariates do not fully explain.",
            _wrap(fig),
        ))

    # 6. Residuals by income
    fig = _fig_residuals_by_income(wase_df)
    if fig:
        inc_df = (
            wase_df[wase_df["data_type"] == "Validated"]
            .dropna(subset=["resid_log", "income_group"])
            .groupby(["pred_type", "income_group"])
            .apply(lambda g: pd.Series(compute_metrics_df(g)))
            .reset_index()
        )
        if len(inc_df) > 0:
            inc_df.to_parquet(OUT_DIR / "wase_diag_income_metrics.parquet", index=False)
        sections.append(_section(
            "residuals_by_income",
            "Log Residual Distribution by Income Group",
            "Box plots of log residuals per income group. "
            "Systematic bias in low-income countries may reflect data sparsity "
            "or structural features not captured by the current covariate set.",
            _wrap(fig),
        ))

    # 7. Percentile calibration
    fig = _fig_percentile_calibration(wase_df)
    if fig:
        pct_df = (
            wase_df[wase_df["data_type"] == "Validated"]
            .dropna(subset=["resid_log", "percentile"])
            .groupby(["pred_type", "percentile"])["resid_log"]
            .agg(mean="mean", sd="std", n="count")
            .reset_index()
        )
        pct_df.to_parquet(OUT_DIR / "wase_diag_percentile_metrics.parquet", index=False)
        sections.append(_section(
            "percentile_calibration",
            "Per-Percentile Bias",
            "Mean log residual ± 1 SE at each percentile. "
            "In WASE, this reflects how well the log-logistic shape parameter "
            "captures the full distributional spread. "
            "Tail bias (at p5 or p95) indicates shape mis-specification.",
            _wrap(fig),
        ))

    # 8. Income × percentile heatmap
    fig = _fig_income_percentile_heatmap(wase_df)
    if fig:
        sections.append(_section(
            "income_percentile_heatmap",
            "Mean Log Residual Heatmap: Income Group × Percentile",
            "Two-dimensional bias surface showing where systematic residuals "
            "concentrate. Red = over-prediction; blue = under-prediction. "
            "Diagonal patterns suggest the shape parameter behaves differently "
            "by country wealth level.",
            _wrap(fig),
        ))

    # 9. Location–shape scatter
    fig = _fig_location_shape_scatter(wase_df)
    if fig:
        sections.append(_section(
            "location_shape_scatter",
            "Residual vs Observed Consumption Level (Location–Shape Diagnostic)",
            "Scatter of log residual against observed log consumption, "
            "coloured by percentile. The smoothed mean line (rolling average) "
            "reveals non-linear patterns in how residuals vary across the "
            "consumption distribution — a key diagnostic for shape mis-specification.",
            _wrap(fig),
        ))

    # 10. Horizon metrics
    result = _fig_horizon_metrics(wase_df)
    if result and result[0] is not None:
        fig, h_df = result
        h_df.to_parquet(OUT_DIR / "wase_diag_horizon_metrics.parquet", index=False)
        sections.append(_section(
            "horizon_metrics",
            "Performance by Forecast Horizon",
            "R², MAE, and MAPE for each forecast horizon (0 = nowcast, 1–5 = forecast). "
            "WASE uses structural indicators only, so its performance should degrade "
            "less steeply with horizon than USE (which propagates GDP uncertainty). "
            "Large drops may indicate regime-sensitive structural features.",
            _wrap(fig),
        ))

    # 11. Country MAE
    result = _fig_country_mae(wase_df)
    if result and result[0] is not None:
        fig, country_df = result
        country_df.to_parquet(OUT_DIR / "wase_diag_country_mae.parquet", index=False)
        sections.append(_section(
            "country_mae",
            "Top 40 Countries by Log-Consumption MAE (Validated)",
            "Countries ranked by mean absolute log-consumption error. "
            "Colour encodes World Bank region. These structurally difficult "
            "countries are likely candidates for manual inspection or "
            "country-specific prior adjustments.",
            _wrap(fig),
        ))

    # ── Write report ───────────────────────────────────────────────
    print("\nAssembling HTML report...")
    build_report(sections, OUT_DIR / "wase_diagnostics.html")

    n_parquet = len(list(OUT_DIR.glob("*.parquet")))
    print(f"\nDONE — {n_parquet} parquet files + HTML report in:\n  {OUT_DIR}")


if __name__ == "__main__":
    main()

Conφ — WASE DIAGNOSTIC REPORT
Output directory: /content/drive/MyDrive/conphi/outputs/conphi_v1_report/diagnostics_wase

Loading data...
  WASE rows from fact table: 163,155
  WASE parameter rows loaded: 7,406

Writing parquet tables...
  wase_diag_params_forest.parquet        : 7,406 rows
  wase_diag_residuals.parquet            : 41,880 rows

Building figures...

Assembling HTML report...
  Report written → /content/drive/MyDrive/conphi/outputs/conphi_v1_report/diagnostics_wase/wase_diagnostics.html

DONE — 9 parquet files + HTML report in:
  /content/drive/MyDrive/conphi/outputs/conphi_v1_report/diagnostics_wase


In [18]:
# Table of survey counts

from pathlib import Path
import pandas as pd

BASE_DIR  = Path("/content/drive/MyDrive/conphi")
INPUT_DIR = BASE_DIR / "data - model inputs"
VERSION   = "v1"

rows = []
for target_year in range(2015, 2026):
    fp = INPUT_DIR / f"conphi_{VERSION}_features_{target_year}.parquet"
    if not fp.exists():
        continue

    df = pd.read_parquet(fp)

    # Filter to national consumption surveys with observed outcomes before target year
    mask = (
        (df["welfare_type"]    == "consumption") &
        (df["year"]             < target_year) &
        (df["log_cons"].notna()) &
        (df["comparable_spell_raw"].notna())
    )
    if "reporting_level" in df.columns:
        mask &= (df["reporting_level"] == "national")

    df = df[mask].copy()

    n_surveys   = df[["iso", "year"]].drop_duplicates().shape[0]
    n_countries = df["iso"].nunique()

    rows.append({
        "Target Year":         target_year,
        "Surveys in Training": n_surveys,
        "Countries Covered":   n_countries,
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

print("\nMarkdown table:")
print("| Target Year | Surveys in Training | Countries Covered |")
print("|:-----------:|:-------------------:|:-----------------:|")
for _, r in results.iterrows():
    print(f"| {int(r['Target Year'])} | {int(r['Surveys in Training'])} | {int(r['Countries Covered'])} |")

 Target Year  Surveys in Training  Countries Covered
        2015                  687                113
        2016                  723                115
        2017                  754                118
        2018                  779                118
        2019                  818                119
        2020                  855                121
        2021                  873                121
        2022                  900                121
        2023                  919                123
        2024                  927                123
        2025                  928                123

Markdown table:
| Target Year | Surveys in Training | Countries Covered |
|:-----------:|:-------------------:|:-----------------:|
| 2015 | 687 | 113 |
| 2016 | 723 | 115 |
| 2017 | 754 | 118 |
| 2018 | 779 | 118 |
| 2019 | 818 | 119 |
| 2020 | 855 | 121 |
| 2021 | 873 | 121 |
| 2022 | 900 | 121 |
| 2023 | 919 | 123 |
| 2024 | 927 | 123 |
| 2025 | 928 | 123 |


In [19]:
# Rows where Country name == Country Code (name was never populated)
same_as_code = country_dim[country_dim["Country"] == country_dim["Country Code"]]
print(f"{len(same_as_code)} countries with name == code:")
print(same_as_code[["Country Code", "Country"]].to_string())

5 countries with name == code:
   Country Code Country
11          BLZ     BLZ
12          BRB     BRB
37          GRD     GRD
54          LCA     LCA
99          SYC     SYC
